In [1]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.utils.path import RAW_STATE_NY_PATH
df_raw = pd.read_csv(RAW_STATE_NY_PATH,dtype='string',keep_default_na=False)

assert ( df_raw.shape[1] == 99)
assert ( df_raw.columns.is_unique)
df_raw.shape

(383614, 99)

In [2]:
TARGET_MAPPING = {
    '1': 1,
    '2': 1,
    '3': 0,
}

eligiable_mask = df_raw['action_taken'].isin(TARGET_MAPPING)

dataset_b = df_raw.loc[eligiable_mask].copy()
dataset_b['y_approved'] = dataset_b['action_taken'].map(TARGET_MAPPING)

cohort_summary = pd.Series({
    'raw_rows':len(df_raw),
    'dataset_b_rows':len(dataset_b),
    'excluded_row':int((~eligiable_mask).sum())
})

cohort_summary

raw_rows          383614
dataset_b_rows    293301
excluded_row       90313
dtype: int64

In [3]:
assert dataset_b['y_approved'].notna().all()
assert set(dataset_b['action_taken'].unique()) == {'1','2','3'}
assert set(dataset_b['y_approved'].unique()) == {0,1}
assert dataset_b.loc[
    dataset_b['action_taken'].isin(['1','2']),
    'y_approved',
].eq(1).all()
assert dataset_b.loc[
    dataset_b["action_taken"].eq("3"),
    "y_approved",
].eq(0).all()

target_check = pd.crosstab(
    dataset_b["action_taken"],
    dataset_b["y_approved"],
    margins=True,
)

target_check

y_approved,0,1,All
action_taken,,,
1,0,208818,208818
2,0,9942,9942
3,74541,0,74541
All,74541,218760,293301


## 5. Feature cần kiểm tra thêm

### 5.1. Kênh nộp hồ sơ và quan hệ tổ chức

Nhóm này kiểm tra bốn candidate features:

- `preapproval`
- `submission_of_application`
- `initially_payable_to_institution`
- `conforming_loan_limit`

Mục tiêu là xác định liệu các feature có cung cấp thông tin hợp lệ,
có trước quyết định và khác với nhóm feature ưu tiên, hay chủ yếu phản
ánh quy trình tiếp nhận, cách tổ chức báo cáo hoặc danh tính tổ chức.

Các kiểm tra trong phần này gồm:

1. raw-token inventory và valid-code coverage;
2. informative, not-applicable, `Exempt`, undetermined và blank;
3. category support và mức độ mất cân bằng;
4. association trong block và với các feature ưu tiên liên quan;
5. concentration theo `lei` như organization-proxy diagnostic.

Phần này chưa thực hiện preprocessing, imputation, encoding, model
training, protected-group sensitivity hoặc quyết định feature cuối cùng.
Không sử dụng `lei` làm predictor.

In [4]:
import numpy as np
from itertools import combinations

GROUP_5_1_COLUMNS = [
    "preapproval",
    "submission_of_application",
    "initially_payable_to_institution",
    "conforming_loan_limit",
]

CONTEXT_5_1_COLUMNS = [
    "loan_purpose",
    "loan_type",
    "lien_status",
    "loan_amount",
]

REFERENCE_5_1_COLUMNS = [
    "lei",
]

required_columns_5_1 = (
    GROUP_5_1_COLUMNS
    + CONTEXT_5_1_COLUMNS
    + REFERENCE_5_1_COLUMNS
)

missing_columns_5_1 = [
    column
    for column in required_columns_5_1
    if column not in dataset_b.columns
]

assert not missing_columns_5_1, (
    f"Missing columns: {missing_columns_5_1}"
)

group_5_1_data = dataset_b.loc[
    :,
    required_columns_5_1,
].copy()

raw_token_inventory_5_1 = pd.concat(
    [
        (
            group_5_1_data[column]
            .value_counts(dropna=False)
            .rename_axis("raw_code")
            .reset_index(name="n")
            .assign(feature=column)
        )
        for column in GROUP_5_1_COLUMNS
    ],
    ignore_index=True,
)[
    ["feature", "raw_code", "n"]
]

raw_token_inventory_5_1

,feature,raw_code,n
0,preapproval,2,283785
1,preapproval,1,9516
2,submission_of_application,1,245493
3,submission_of_application,2,39534
4,submission_of_application,1111,8274
5,initially_payable_to_institution,1,269984
6,initially_payable_to_institution,2,9510
7,initially_payable_to_institution,1111,8274
8,initially_payable_to_institution,3,5533
9,conforming_loan_limit,C,280700


In [5]:
EXPECTED_CODE_LABELS_5_1 = {
    "preapproval": {
        "1": "Preapproval requested",
        "2": "Preapproval not requested",
        "3": "Not applicable",
    },
    "submission_of_application": {
        "1": "Submitted directly to reporting institution",
        "2": "Not submitted directly to reporting institution",
        "3": "Not applicable",
        "1111": "Exempt",
    },
    "initially_payable_to_institution": {
        "1": "Initially payable to reporting institution",
        "2": "Not initially payable to reporting institution",
        "3": "Not applicable",
        "1111": "Exempt",
    },
    "conforming_loan_limit": {
        "C": "Conforming",
        "NC": "Non-conforming",
        "U": "Undetermined",
    },
}

SPECIAL_CODES_5_1 = {
    "preapproval": {"3"},
    "submission_of_application": {"3", "1111"},
    "initially_payable_to_institution": {"3", "1111"},
    "conforming_loan_limit": {"U"},
}

schema_definition_5_1 = pd.Series({
    "feature_count": len(EXPECTED_CODE_LABELS_5_1),
    "all_features_have_special_state_rules": (
        set(EXPECTED_CODE_LABELS_5_1) == set(SPECIAL_CODES_5_1)
    ),
})
schema_definition_5_1


feature_count                               4
all_features_have_special_state_rules    True
dtype: object

In [6]:
def audit_feature_codes_5_1(data, feature):
    values = data[feature]

    code_labels = EXPECTED_CODE_LABELS_5_1[feature]
    special_codes = SPECIAL_CODES_5_1[feature]

    audit_rows = []

    for raw_code, count in values.value_counts(
        dropna=False
    ).items():
        blank_or_missing = (
            pd.isna(raw_code)
            or str(raw_code).strip() == ""
        )

        if blank_or_missing:
            description = "Blank or missing"
            status = "blank_or_missing"
        elif raw_code not in code_labels:
            description = "Unrecognized code"
            status = "invalid_code"
        elif raw_code in special_codes:
            description = code_labels[raw_code]
            status = "valid_special"
        else:
            description = code_labels[raw_code]
            status = "valid_informative"

        audit_rows.append({
            "feature": feature,
            "raw_code": raw_code,
            "description": description,
            "status": status,
            "n": int(count),
            "share_pct": round(
                count / len(values) * 100,
                4,
            ),
        })

    return pd.DataFrame(audit_rows)


audit_5_1 = pd.concat(
    [
        audit_feature_codes_5_1(
            data=group_5_1_data,
            feature=feature,
        )
        for feature in GROUP_5_1_COLUMNS
    ],
    ignore_index=True,
)

quality_rows_5_1 = []

for feature in GROUP_5_1_COLUMNS:
    feature_audit = audit_5_1.loc[
        audit_5_1["feature"] == feature
    ]

    informative_n = int(
        feature_audit.loc[
            feature_audit["status"] == "valid_informative",
            "n",
        ].sum()
    )

    special_n = int(
        feature_audit.loc[
            feature_audit["status"] == "valid_special",
            "n",
        ].sum()
    )

    blank_or_missing_n = int(
        feature_audit.loc[
            feature_audit["status"] == "blank_or_missing",
            "n",
        ].sum()
    )

    invalid_code_n = int(
        feature_audit.loc[
            feature_audit["status"] == "invalid_code",
            "n",
        ].sum()
    )

    valid_n = informative_n + special_n

    quality_rows_5_1.append({
        "feature": feature,
        "row_count": len(group_5_1_data),
        "valid_n": valid_n,
        "valid_pct": round(
            valid_n / len(group_5_1_data) * 100,
            4,
        ),
        "informative_n": informative_n,
        "informative_pct": round(
            informative_n / len(group_5_1_data) * 100,
            4,
        ),
        "special_n": special_n,
        "special_pct": round(
            special_n / len(group_5_1_data) * 100,
            4,
        ),
        "blank_or_missing_n": blank_or_missing_n,
        "invalid_code_n": invalid_code_n,
        "raw_unique_count": (
            group_5_1_data[feature]
            .nunique(dropna=False)
        ),
    })

quality_5_1 = pd.DataFrame(quality_rows_5_1)

invalid_codes_5_1 = audit_5_1.loc[
    audit_5_1["status"] == "invalid_code"
].copy()

display(quality_5_1)
display(audit_5_1)
display(invalid_codes_5_1)

,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,special_n,special_pct,blank_or_missing_n,invalid_code_n,raw_unique_count
0,preapproval,293301,293301,100.0000,293301,100.0000,0,0.0000,0,0,2
1,submission_of_application,293301,293301,100.0000,285027,97.1790,8274,2.8210,0,0,3
2,initially_payable_to_institution,293301,293301,100.0000,279494,95.2925,13807,4.7075,0,0,4
3,conforming_loan_limit,293301,290194,98.9407,290136,98.9209,58,0.0198,0,3107,4


,feature,raw_code,description,status,n,share_pct
0,preapproval,2,Preapproval not requested,valid_informative,283785,96.7556
1,preapproval,1,Preapproval requested,valid_informative,9516,3.2444
2,submission_of_application,1,Submitted directly to reporting institution,valid_informative,245493,83.7000
3,submission_of_application,2,Not submitted directly to reporting institution,valid_informative,39534,13.4790
4,submission_of_application,1111,Exempt,valid_special,8274,2.8210
5,initially_payable_to_institution,1,Initially payable to reporting institution,valid_informative,269984,92.0501
6,initially_payable_to_institution,2,Not initially payable to reporting institution,valid_informative,9510,3.2424
7,initially_payable_to_institution,1111,Exempt,valid_special,8274,2.8210
8,initially_payable_to_institution,3,Not applicable,valid_special,5533,1.8865
9,conforming_loan_limit,C,Conforming,valid_informative,280700,95.7037


,feature,raw_code,description,status,n,share_pct
11,conforming_loan_limit,NA,Unrecognized code,invalid_code,3107,1.0593


In [7]:
assert len(group_5_1_data) == len(dataset_b)

assert quality_5_1["row_count"].eq(
    len(dataset_b)
).all()

quality_reconciled_5_1 = (
    quality_5_1["valid_n"]
    + quality_5_1["blank_or_missing_n"]
    + quality_5_1["invalid_code_n"]
)

assert quality_reconciled_5_1.eq(
    len(dataset_b)
).all()

audit_totals_5_1 = (
    audit_5_1
    .groupby("feature")["n"]
    .sum()
)

assert audit_totals_5_1.eq(
    len(dataset_b)
).all()

validation_5_1 = pd.Series({
    "row_count_reconciled": True,
    "quality_counts_reconciled": True,
    "audit_counts_reconciled": True,
    "schema_map_matches_observed_tokens": (
        invalid_codes_5_1.empty
    ),
})

validation_5_1

row_count_reconciled                   True
quality_counts_reconciled              True
audit_counts_reconciled                True
schema_map_matches_observed_tokens    False
dtype: bool

In [8]:
def cramers_v_5_1(data,left,right):
    contingency = pd.crosstab(
        data[left],
        data[right],
        dropna=False,
    )

    observed = contingency.to_numpy(
        dtype=float
    )

    n = observed.sum()
    rows,columns = observed.shape

    if(
        n == 0
        or rows < 2
        or columns < 2
    ):
        return np.nan

    expected = np.outer(
        observed.sum(axis= 1),
        observed.sum(axis= 0),
    ) /n 

    chi_square_terms = np.divide(
        (observed - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected > 0,
    )

    chi_square = chi_square_terms.sum()

    denominator = min(rows -1, columns - 1,)
    if denominator == 0:
        return np.nan

    return float(
        np.sqrt(
            (chi_square/ n)
            / denominator
        )
    )

association_specs_5_1 = [
    (left,right,'within_5_1_block')
    for left,right in combinations(
        GROUP_5_1_COLUMNS,
        2,
    )
]

association_specs_5_1.extend([
    (
        "preapproval",
        "loan_purpose",
        "priority_feature_context",
    ),
    (
        "conforming_loan_limit",
        "loan_type",
        "priority_feature_context",
    ),
    (
        "conforming_loan_limit",
        "lien_status",
        "priority_feature_context",
    ),
])

association_5_1 = pd.DataFrame(
    [
        {
            "left_feature": left,
            "right_feature": right,
            "scope": scope,
            "cramers_v": round(
                cramers_v_5_1(
                    group_5_1_data,
                    left,
                    right,
                ),
                4,
            ),
        }
        for left, right, scope
        in association_specs_5_1
    ]
)

association_5_1

,left_feature,right_feature,scope,cramers_v
0,preapproval,submission_of_application,within_5_1_block,0.0633
1,preapproval,initially_payable_to_institution,within_5_1_block,0.0288
2,preapproval,conforming_loan_limit,within_5_1_block,0.0560
3,submission_of_application,initially_payable_to_institution,within_5_1_block,0.7178
4,submission_of_application,conforming_loan_limit,within_5_1_block,0.0654
5,initially_payable_to_institution,conforming_loan_limit,within_5_1_block,0.0497
6,preapproval,loan_purpose,priority_feature_context,0.1927
7,conforming_loan_limit,loan_type,priority_feature_context,0.0363
8,conforming_loan_limit,lien_status,priority_feature_context,0.0890


In [9]:
def row_percentage_crosstab_5_1(
    data,
    row_feature,
    column_feature,
):
    counts = pd.crosstab(
        data[row_feature],
        data[column_feature],
        dropna=False,
    )

    row_pct = (
        counts
        .div(counts.sum(axis=1), axis=0)
        .mul(100)
        .round(2)
    )

    return counts, row_pct


key_relationships_5_1 = [
    (
        "submission_of_application",
        "initially_payable_to_institution",
    ),
    (
        "preapproval",
        "loan_purpose",
    ),
    (
        "conforming_loan_limit",
        "loan_type",
    ),
    (
        "conforming_loan_limit",
        "lien_status",
    ),
]

relationship_tables_5_1 = {}

for left, right in key_relationships_5_1:
    counts, row_pct = row_percentage_crosstab_5_1(
        group_5_1_data,
        left,
        right,
    )

    relationship_tables_5_1[
        (left, right)
    ] = {
        "counts": counts,
        "row_pct": row_pct,
    }

    print(f"{left} × {right}")
    display(counts)
    display(row_pct)

submission_of_application × initially_payable_to_institution


initially_payable_to_institution,1,1111,2,3
submission_of_application,,,,
1,236015,0,5184,4294
1111,0,8274,0,0
2,33969,0,4326,1239


initially_payable_to_institution,1,1111,2,3
submission_of_application,,,,
1,96.14,0.0,2.11,1.75
1111,0.00,100.0,0.00,0.00
2,85.92,0.0,10.94,3.13


preapproval × loan_purpose


loan_purpose,1,2,31,32,4,5
preapproval,,,,,,
1,9516,0,0,0,0,0
2,129628,49762,29412,34595,40360,28


loan_purpose,1,2,31,32,4,5
preapproval,,,,,,
1,100.00,0.00,0.00,0.00,0.00,0.00
2,45.68,17.54,10.36,12.19,14.22,0.01


... (2 outputs left out)

loan_type,1,2,3,4
conforming_loan_limit,,,,
C,90.18,7.51,2.20,0.12
NA,99.71,0.29,0.00,0.00
NC,99.05,0.47,0.49,0.00
U,100.00,0.00,0.00,0.00


conforming_loan_limit × lien_status


lien_status,1,2
conforming_loan_limit,,
C,193289,87411
NA,3008,99
NC,8082,1354
U,52,6


lien_status,1,2
conforming_loan_limit,,
C,68.86,31.14
NA,96.81,3.19
NC,85.65,14.35
U,89.66,10.34


In [10]:
conforming_amount_data_5_1 = (
    group_5_1_data[
        [
            "conforming_loan_limit",
            "loan_amount",
        ]
    ]
    .copy()
)

conforming_amount_data_5_1[
    "loan_amount_numeric"
] = pd.to_numeric(
    conforming_amount_data_5_1[
        "loan_amount"
    ],
    errors="coerce",
)

conforming_amount_profile_5_1 = (
    conforming_amount_data_5_1
    .groupby(
        "conforming_loan_limit",
        dropna=False,
    )
    .agg(
        row_count=(
            "loan_amount",
            "size",
        ),
        numeric_n=(
            "loan_amount_numeric",
            "count",
        ),
        q1=(
            "loan_amount_numeric",
            lambda values: values.quantile(0.25),
        ),
        median=(
            "loan_amount_numeric",
            "median",
        ),
        q3=(
            "loan_amount_numeric",
            lambda values: values.quantile(0.75),
        ),
    )
    .reset_index()
)

conforming_amount_profile_5_1[
    "numeric_pct"
] = (
    conforming_amount_profile_5_1[
        "numeric_n"
    ]
    .div(
        conforming_amount_profile_5_1[
            "row_count"
        ]
    )
    .mul(100)
    .round(2)
)

conforming_amount_profile_5_1

,conforming_loan_limit,row_count,numeric_n,q1,median,q3,numeric_pct
0,C,280700,280700,95000.0,195000.0,395000.0,100.0
1,NA,3107,3107,805000.0,1825000.0,4405000.0,100.0
2,NC,9436,9436,1205000.0,1505000.0,2065000.0,100.0
3,U,58,58,825000.0,955000.0,1085000.0,100.0


In [11]:
def lei_proxy_diagnostic_5_1(
    data,
    features,
):
    lei_values = data["lei"]

    valid_lei_mask = (
        lei_values.notna()
        & lei_values.str.strip().ne("")
    )

    proxy_data = data.loc[
        valid_lei_mask,
        ["lei"] + features,
    ].copy()

    summary_rows = []
    detail_frames = []

    for feature in features:
        counts = (
            proxy_data
            .groupby(
                ["lei", feature],
                dropna=False,
            )
            .size()
            .rename("n")
            .reset_index()
        )

        counts["lei_total_n"] = (
            counts
            .groupby("lei")["n"]
            .transform("sum")
        )

        counts["within_lei_share"] = (
            counts["n"]
            / counts["lei_total_n"]
        )

        per_lei = (
            counts
            .groupby("lei")
            .agg(
                row_count=("n", "sum"),
                category_count=(
                    feature,
                    "nunique",
                ),
                dominant_share=(
                    "within_lei_share",
                    "max",
                ),
            )
            .reset_index()
        )

        per_lei.insert(
            0,
            "feature",
            feature,
        )

        detail_frames.append(per_lei)

        weighted_dominant_share = np.average(
            per_lei["dominant_share"],
            weights=per_lei["row_count"],
        )

        single_category_lei_n = int(
            per_lei["category_count"]
            .eq(1)
            .sum()
        )

        summary_rows.append({
            "feature": feature,
            "scope": "full_cohort_descriptive_only",
            "lei_n": len(per_lei),
            "blank_lei_row_n": int(
                (~valid_lei_mask).sum()
            ),
            "single_category_lei_n": (
                single_category_lei_n
            ),
            "single_category_lei_pct": round(
                single_category_lei_n
                / len(per_lei)
                * 100,
                2,
            ),
            "median_dominant_share_pct": round(
                per_lei["dominant_share"]
                .median()
                * 100,
                2,
            ),
            "row_weighted_dominant_share_pct": round(
                weighted_dominant_share
                * 100,
                2,
            ),
        })

    return (
        pd.DataFrame(summary_rows),
        pd.concat(
            detail_frames,
            ignore_index=True,
        ),
    )


(
    lei_proxy_summary_5_1,
    lei_proxy_detail_5_1,
) = lei_proxy_diagnostic_5_1(
    data=group_5_1_data,
    features=GROUP_5_1_COLUMNS,
)

lei_proxy_summary_5_1

,feature,scope,lei_n,blank_lei_row_n,single_category_lei_n,single_category_lei_pct,median_dominant_share_pct,row_weighted_dominant_share_pct
0,preapproval,full_cohort_descriptive_only,715,0,588,82.24,100.0,97.34
1,submission_of_application,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.30
2,initially_payable_to_institution,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.57
3,conforming_loan_limit,full_cohort_descriptive_only,715,0,377,52.73,100.0,95.93


In [12]:
quality_lookup_5_1 = (
    quality_5_1
    .set_index("feature")
)

decision_5_1 = pd.DataFrame({
    "feature_or_block": GROUP_5_1_COLUMNS,
})

decision_5_1["business_reason"] = (
    decision_5_1["feature_or_block"].map({
        "preapproval": (
            "Represents whether preapproval was requested."
        ),
        "submission_of_application": (
            "Represents the application submission channel."
        ),
        "initially_payable_to_institution": (
            "Represents the initial institutional relationship."
        ),
        "conforming_loan_limit": (
            "Represents reported conforming-loan-limit status."
        ),
    })
)

decision_5_1["timing_status"] = (
    "requires_lineage_confirmation"
)

decision_5_1["quality_evidence"] = (
    decision_5_1["feature_or_block"].map({
        feature: (
            f"{quality_lookup_5_1.loc[feature, 'valid_pct']:.2f}% "
            f"valid; "
            f"{quality_lookup_5_1.loc[feature, 'informative_pct']:.2f}% "
            f"informative; "
            f"{int(quality_lookup_5_1.loc[feature, 'invalid_code_n'])} "
            "invalid codes."
        )
        for feature in GROUP_5_1_COLUMNS
    })
)

decision_5_1["redundancy_evidence"] = (
    "Review association_5_1 and relationship_tables_5_1."
)

decision_5_1["organization_proxy_evidence"] = (
    "Review lei_proxy_summary_5_1; descriptive only."
)

decision_5_1["protected_group_sensitivity"] = (
    "not_run"
)

decision_5_1["screening_status"] = (
    "pending_train_validation_evidence"
)

decision_5_1["decision"] = pd.NA

decision_5_1["decision_reason"] = (
    "Pending lineage confirmation, train-only analysis, "
    "and grouped addition/ablation evidence."
)

decision_5_1["gsv_player"] = pd.NA

decision_5_1

,feature_or_block,business_reason,timing_status,quality_evidence,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player
0,preapproval,Represents whether preapproval was requested.,requires_lineage_confirmation,100.00% valid; 100.00% informative; 0 invalid ...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>
1,submission_of_application,Represents the application submission channel.,requires_lineage_confirmation,100.00% valid; 97.18% informative; 0 invalid c...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>
2,initially_payable_to_institution,Represents the initial institutional relations...,requires_lineage_confirmation,100.00% valid; 95.29% informative; 0 invalid c...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>
3,conforming_loan_limit,Represents reported conforming-loan-limit status.,requires_lineage_confirmation,98.94% valid; 98.92% informative; 3107 invalid...,Review association_5_1 and relationship_tables...,Review lei_proxy_summary_5_1; descriptive only.,not_run,pending_train_validation_evidence,<NA>,"Pending lineage confirmation, train-only analy...",<NA>


### 5.1.1. Bổ sung bằng chứng để đóng screening

Các cell dưới đây **chỉ bổ sung** audit cho phần 5.1; không thay thế hoặc xóa các cell trước. Mục tiêu là sửa phân loại schema của `conforming_loan_limit`, tách association trên full population và informative-only population, làm rõ phạm vi áp dụng của `preapproval`, bổ sung LEI baseline/size sensitivity và tạo screening decision record.

Nguồn đối chiếu:

- `docs/handbook/contracts/feature_governance.md` và `docs/handbook/contracts/data_contract.md`: yêu cầu tách riêng `NA`, `Exempt`/`1111`, not-applicable; kiểm tra timing, conditional applicability, redundancy và organization proxy.
- [2024 HMDA Filing Instructions Guide](https://files.ffiec.cfpb.gov/documentation/2024-hmda-fig.pdf): định nghĩa `preapproval`, `submission_of_application` và `initially_payable_to_institution`.
- [CFPB Public LAR Code Sheet](https://s3.amazonaws.com/cfpb-hmda-public/prod/help/2018-public-LAR-code-sheet.pdf): `conforming_loan_limit` có bốn trạng thái hợp lệ `C`, `NC`, `U`, `NA`; `NA` là *Not Applicable*.
- [CFPB HMDA disclosure policy guidance](https://files.consumerfinance.gov/f/documents/HMDA_Data_Disclosure_Policy_Guidance.Executive_Summary.FINAL.12212018.pdf): conforming-loan-limit là public derived indicator dựa trên reported loan amount và GSE limit.

Ba trường về preapproval/application channel mô tả thông tin của yêu cầu hoặc hồ sơ. Riêng `conforming_loan_limit` là trường public-derived có thể dùng giới hạn tại thời điểm application **hoặc** origination; vì vậy chưa gắn nhãn an toàn trước quyết định cho mô hình approval cho đến khi lineage của từng action type được xác nhận.

In [13]:
EXPECTED_CODE_LABELS_5_1_REVIEWED = {
    **EXPECTED_CODE_LABELS_5_1,
    "conforming_loan_limit": {
        "C": "Conforming",
        "NC": "Non-conforming",
        "U": "Undetermined",
        "NA": "Not applicable",
    },
}

SPECIAL_CODES_5_1_REVIEWED = {
    **SPECIAL_CODES_5_1,
    "conforming_loan_limit": {"U", "NA"},
}

def audit_feature_codes_5_1_reviewed(data, feature):
    values = data[feature]
    code_labels = EXPECTED_CODE_LABELS_5_1_REVIEWED[feature]
    special_codes = SPECIAL_CODES_5_1_REVIEWED[feature]
    audit_rows = []

    for raw_code, count in values.value_counts(dropna=False).items():
        blank_or_missing = (
            pd.isna(raw_code)
            or str(raw_code).strip() == ""
        )

        if blank_or_missing:
            description = "Blank or missing"
            status = "blank_or_missing"
        elif raw_code not in code_labels:
            description = "Unrecognized code"
            status = "invalid_code"
        elif raw_code in special_codes:
            description = code_labels[raw_code]
            status = "valid_special"
        else:
            description = code_labels[raw_code]
            status = "valid_informative"

        audit_rows.append({
            "feature": feature,
            "raw_code": raw_code,
            "description": description,
            "status": status,
            "n": int(count),
            "share_pct": round(count / len(values) * 100, 4),
        })

    return pd.DataFrame(audit_rows)

audit_5_1_reviewed = pd.concat(
    [
        audit_feature_codes_5_1_reviewed(
            data=group_5_1_data,
            feature=feature,
        )
        for feature in GROUP_5_1_COLUMNS
    ],
    ignore_index=True,
)

quality_rows_5_1_reviewed = []
for feature in GROUP_5_1_COLUMNS:
    feature_audit = audit_5_1_reviewed.loc[
        audit_5_1_reviewed["feature"].eq(feature)
    ]
    informative_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("valid_informative"),
            "n",
        ].sum()
    )
    special_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("valid_special"),
            "n",
        ].sum()
    )
    blank_or_missing_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("blank_or_missing"),
            "n",
        ].sum()
    )
    invalid_code_n = int(
        feature_audit.loc[
            feature_audit["status"].eq("invalid_code"),
            "n",
        ].sum()
    )
    valid_n = informative_n + special_n

    quality_rows_5_1_reviewed.append({
        "feature": feature,
        "row_count": len(group_5_1_data),
        "valid_n": valid_n,
        "valid_pct": round(valid_n / len(group_5_1_data) * 100, 4),
        "informative_n": informative_n,
        "informative_pct": round(
            informative_n / len(group_5_1_data) * 100,
            4,
        ),
        "special_n": special_n,
        "special_pct": round(
            special_n / len(group_5_1_data) * 100,
            4,
        ),
        "blank_or_missing_n": blank_or_missing_n,
        "invalid_code_n": invalid_code_n,
        "raw_unique_count": group_5_1_data[feature].nunique(
            dropna=False
        ),
    })

quality_5_1_reviewed = pd.DataFrame(quality_rows_5_1_reviewed)
invalid_codes_5_1_reviewed = audit_5_1_reviewed.loc[
    audit_5_1_reviewed["status"].eq("invalid_code")
].copy()

validation_5_1_reviewed = pd.Series({
    "row_count_reconciled": quality_5_1_reviewed[
        "row_count"
    ].eq(len(dataset_b)).all(),
    "audit_counts_reconciled": audit_5_1_reviewed.groupby(
        "feature"
    )["n"].sum().eq(len(dataset_b)).all(),
    "schema_map_matches_observed_tokens": (
        invalid_codes_5_1_reviewed.empty
    ),
})

assert validation_5_1_reviewed.all()
display(quality_5_1_reviewed)
display(audit_5_1_reviewed)
display(validation_5_1_reviewed)

,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,special_n,special_pct,blank_or_missing_n,invalid_code_n,raw_unique_count
0,preapproval,293301,293301,100.0,293301,100.0000,0,0.0000,0,0,2
1,submission_of_application,293301,293301,100.0,285027,97.1790,8274,2.8210,0,0,3
2,initially_payable_to_institution,293301,293301,100.0,279494,95.2925,13807,4.7075,0,0,4
3,conforming_loan_limit,293301,293301,100.0,290136,98.9209,3165,1.0791,0,0,4


,feature,raw_code,description,status,n,share_pct
0,preapproval,2,Preapproval not requested,valid_informative,283785,96.7556
1,preapproval,1,Preapproval requested,valid_informative,9516,3.2444
2,submission_of_application,1,Submitted directly to reporting institution,valid_informative,245493,83.7000
3,submission_of_application,2,Not submitted directly to reporting institution,valid_informative,39534,13.4790
4,submission_of_application,1111,Exempt,valid_special,8274,2.8210
5,initially_payable_to_institution,1,Initially payable to reporting institution,valid_informative,269984,92.0501
6,initially_payable_to_institution,2,Not initially payable to reporting institution,valid_informative,9510,3.2424
7,initially_payable_to_institution,1111,Exempt,valid_special,8274,2.8210
8,initially_payable_to_institution,3,Not applicable,valid_special,5533,1.8865
9,conforming_loan_limit,C,Conforming,valid_informative,280700,95.7037


row_count_reconciled                  True
audit_counts_reconciled               True
schema_map_matches_observed_tokens    True
dtype: bool

In [14]:
submission_informative_codes_5_1 = {"1", "2"}
initial_payable_informative_codes_5_1 = {"1", "2"}

informative_pair_mask_5_1 = (
    group_5_1_data["submission_of_application"].isin(
        submission_informative_codes_5_1
    )
    & group_5_1_data["initially_payable_to_institution"].isin(
        initial_payable_informative_codes_5_1
    )
)

informative_pair_data_5_1 = group_5_1_data.loc[
    informative_pair_mask_5_1
].copy()

informative_pair_counts_5_1, informative_pair_row_pct_5_1 = (
    row_percentage_crosstab_5_1(
        informative_pair_data_5_1,
        "submission_of_application",
        "initially_payable_to_institution",
    )
)

full_pair_cramers_v_5_1 = cramers_v_5_1(
    group_5_1_data,
    "submission_of_application",
    "initially_payable_to_institution",
)
informative_pair_cramers_v_5_1 = cramers_v_5_1(
    informative_pair_data_5_1,
    "submission_of_application",
    "initially_payable_to_institution",
)

association_population_comparison_5_1 = pd.DataFrame([
    {
        "population": "full_cohort",
        "n": len(group_5_1_data),
        "cramers_v": round(full_pair_cramers_v_5_1, 4),
    },
    {
        "population": "informative_only_codes_1_2",
        "n": len(informative_pair_data_5_1),
        "cramers_v": round(informative_pair_cramers_v_5_1, 4),
    },
])

preapproval_requested_mask_5_1 = group_5_1_data[
    "preapproval"
].eq("1")
home_purchase_mask_5_1 = group_5_1_data["loan_purpose"].eq("1")
requested_preapproval_n_5_1 = int(preapproval_requested_mask_5_1.sum())
home_purchase_n_5_1 = int(home_purchase_mask_5_1.sum())
requested_home_purchase_n_5_1 = int(
    (preapproval_requested_mask_5_1 & home_purchase_mask_5_1).sum()
)

preapproval_applicability_5_1 = pd.Series({
    "requested_preapproval_n": requested_preapproval_n_5_1,
    "requested_home_purchase_n": requested_home_purchase_n_5_1,
    "requested_within_home_purchase_pct": round(
        requested_home_purchase_n_5_1
        / requested_preapproval_n_5_1
        * 100,
        4,
    ),
    "home_purchase_n": home_purchase_n_5_1,
    "requested_share_of_home_purchase_pct": round(
        requested_home_purchase_n_5_1
        / home_purchase_n_5_1
        * 100,
        4,
    ),
})

assert len(informative_pair_data_5_1) > 0
assert requested_preapproval_n_5_1 > 0
assert requested_home_purchase_n_5_1 == requested_preapproval_n_5_1

display(association_population_comparison_5_1)
display(informative_pair_counts_5_1)
display(informative_pair_row_pct_5_1)
display(preapproval_applicability_5_1)

,population,n,cramers_v
0,full_cohort,293301,0.7178
1,informative_only_codes_1_2,279494,0.1735


initially_payable_to_institution,1,2
submission_of_application,,
1,236015,5184
2,33969,4326


initially_payable_to_institution,1,2
submission_of_application,,
1,97.85,2.15
2,88.70,11.30


requested_preapproval_n                   9516.000
requested_home_purchase_n                 9516.000
requested_within_home_purchase_pct         100.000
home_purchase_n                         139144.000
requested_share_of_home_purchase_pct         6.839
dtype: float64

In [15]:
global_dominance_rows_5_1 = []
for feature in GROUP_5_1_COLUMNS:
    feature_counts = group_5_1_data[feature].value_counts(dropna=False)
    global_dominance_rows_5_1.append({
        "feature": feature,
        "global_dominant_share_pct": round(
            feature_counts.iloc[0] / len(group_5_1_data) * 100,
            4,
        ),
    })

global_dominance_5_1 = pd.DataFrame(global_dominance_rows_5_1)
lei_proxy_enhanced_5_1 = lei_proxy_summary_5_1.merge(
    global_dominance_5_1,
    on="feature",
    validate="one_to_one",
)
lei_proxy_enhanced_5_1["lei_uplift_vs_global_pp"] = (
    lei_proxy_enhanced_5_1["row_weighted_dominant_share_pct"]
    - lei_proxy_enhanced_5_1["global_dominant_share_pct"]
).round(4)

lei_size_profile_5_1 = (
    lei_proxy_detail_5_1.loc[
        lei_proxy_detail_5_1["feature"].eq(GROUP_5_1_COLUMNS[0]),
        ["lei", "row_count"],
    ]
    .drop_duplicates("lei")
    .sort_values(["row_count", "lei"])
    .reset_index(drop=True)
)
lei_size_profile_5_1["lei_size_quartile"] = pd.qcut(
    lei_size_profile_5_1["row_count"].rank(method="first"),
    q=4,
    labels=["Q1_smallest", "Q2", "Q3", "Q4_largest"],
)

lei_proxy_detail_with_size_5_1 = lei_proxy_detail_5_1.merge(
    lei_size_profile_5_1[["lei", "lei_size_quartile"]],
    on="lei",
    validate="many_to_one",
)

lei_size_sensitivity_rows_5_1 = []
for feature in GROUP_5_1_COLUMNS:
    feature_detail = lei_proxy_detail_with_size_5_1.loc[
        lei_proxy_detail_with_size_5_1["feature"].eq(feature)
    ]
    for size_quartile, size_detail in feature_detail.groupby(
        "lei_size_quartile",
        observed=True,
    ):
        lei_size_sensitivity_rows_5_1.append({
            "feature": feature,
            "lei_size_quartile": str(size_quartile),
            "lei_n": size_detail["lei"].nunique(),
            "row_n": int(size_detail["row_count"].sum()),
            "row_weighted_dominant_share_pct": round(
                np.average(
                    size_detail["dominant_share"],
                    weights=size_detail["row_count"],
                )
                * 100,
                4,
            ),
        })

lei_size_sensitivity_5_1 = pd.DataFrame(
    lei_size_sensitivity_rows_5_1
)
lei_size_distribution_5_1 = lei_size_profile_5_1[
    "row_count"
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

assert lei_proxy_enhanced_5_1["lei_uplift_vs_global_pp"].notna().all()
assert lei_size_sensitivity_5_1["row_n"].gt(0).all()

display(lei_proxy_enhanced_5_1)
display(lei_size_distribution_5_1)
display(lei_size_sensitivity_5_1)

,feature,scope,lei_n,blank_lei_row_n,single_category_lei_n,single_category_lei_pct,median_dominant_share_pct,row_weighted_dominant_share_pct,global_dominant_share_pct,lei_uplift_vs_global_pp
0,preapproval,full_cohort_descriptive_only,715,0,588,82.24,100.0,97.34,96.7556,0.5844
1,submission_of_application,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.30,83.7000,12.6000
2,initially_payable_to_institution,full_cohort_descriptive_only,715,0,603,84.34,100.0,96.57,92.0501,4.5199
3,conforming_loan_limit,full_cohort_descriptive_only,715,0,377,52.73,100.0,95.93,95.7037,0.2263


count      715.000000
mean       410.211189
std       1392.652345
min          1.000000
25%          2.000000
50%         25.000000
75%        195.500000
90%        893.400000
95%       1818.500000
99%       7554.100000
max      14757.000000
Name: row_count, dtype: float64

,feature,lei_size_quartile,lei_n,row_n,row_weighted_dominant_share_pct
0,preapproval,Q1_smallest,179,226,100.0000
1,preapproval,Q2,179,1715,97.7259
2,preapproval,Q3,178,15683,96.0148
3,preapproval,Q4_largest,179,275677,97.4151
4,submission_of_application,Q1_smallest,179,226,98.6726
5,submission_of_application,Q2,179,1715,97.4927
6,submission_of_application,Q3,178,15683,96.3910
7,submission_of_application,Q4_largest,179,275677,96.2830
8,initially_payable_to_institution,Q1_smallest,179,226,98.6726
9,initially_payable_to_institution,Q2,179,1715,97.9009


In [16]:
quality_lookup_5_1_reviewed = quality_5_1_reviewed.set_index("feature")
lei_lookup_5_1_reviewed = lei_proxy_enhanced_5_1.set_index("feature")

timing_evidence_5_1 = pd.DataFrame([
    {
        "feature": "preapproval",
        "timing_status": "available_before_decision_by_definition",
        "timing_evidence": (
            "2024 HMDA FIG 4(a)(4): whether the application involved "
            "a request for preapproval."
        ),
    },
    {
        "feature": "submission_of_application",
        "timing_status": "available_before_decision_by_definition",
        "timing_evidence": (
            "2024 HMDA FIG 4(a)(33): whether the applicant submitted "
            "the application directly to the institution."
        ),
    },
    {
        "feature": "initially_payable_to_institution",
        "timing_status": "available_before_decision_by_definition",
        "timing_evidence": (
            "2024 HMDA FIG 4(a)(33): for an application, whether the "
            "obligation would have been initially payable to the institution."
        ),
    },
    {
        "feature": "conforming_loan_limit",
        "timing_status": "not_consistently_predecision_safe",
        "timing_evidence": (
            "CFPB Regulation C timing: reported loan amount uses the legal-obligation "
            "amount for originated loans, approved amount for action 2, and amount "
            "applied for for denied loans; this public-derived flag is excluded."
        ),
    },
])

screening_decision_5_1 = pd.DataFrame({
    "feature_or_block": GROUP_5_1_COLUMNS,
}).merge(
    timing_evidence_5_1,
    left_on="feature_or_block",
    right_on="feature",
    validate="one_to_one",
).drop(columns="feature")

screening_decision_5_1["quality_evidence"] = (
    screening_decision_5_1["feature_or_block"].map({
        feature: (
            f"{quality_lookup_5_1_reviewed.loc[feature, 'valid_pct']:.2f}% "
            f"valid; "
            f"{quality_lookup_5_1_reviewed.loc[feature, 'informative_pct']:.2f}% "
            f"informative; "
            f"{int(quality_lookup_5_1_reviewed.loc[feature, 'invalid_code_n'])} "
            "invalid codes."
        )
        for feature in GROUP_5_1_COLUMNS
    })
)

screening_decision_5_1["conditional_applicability"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": (
            f"All {requested_preapproval_n_5_1} requested-preapproval rows "
            f"are home purchase; they represent "
            f"{preapproval_applicability_5_1['requested_share_of_home_purchase_pct']:.2f}% "
            "of home-purchase rows."
        ),
        "submission_of_application": "Codes 3 and 1111 retained separately.",
        "initially_payable_to_institution": (
            "Codes 3 and 1111 retained separately."
        ),
        "conforming_loan_limit": "U and NA retained as separate valid special states.",
    })
)

screening_decision_5_1["redundancy_evidence"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": (
            "Conditional on home-purchase scope; not a duplicate of loan_purpose."
        ),
        "submission_of_application": (
            f"Joint block with initially_payable: full V={full_pair_cramers_v_5_1:.4f}; "
            f"informative-only V={informative_pair_cramers_v_5_1:.4f}."
        ),
        "initially_payable_to_institution": (
            f"Joint block with submission: full V={full_pair_cramers_v_5_1:.4f}; "
            f"informative-only V={informative_pair_cramers_v_5_1:.4f}."
        ),
        "conforming_loan_limit": (
            "Public-derived from outcome-dependent reported loan amount; excluded before validation."
        ),
    })
)

screening_decision_5_1["organization_proxy_evidence"] = (
    screening_decision_5_1["feature_or_block"].map({
        feature: (
            f"LEI row-weighted dominance "
            f"{lei_lookup_5_1_reviewed.loc[feature, 'row_weighted_dominant_share_pct']:.2f}%; "
            f"global dominance "
            f"{lei_lookup_5_1_reviewed.loc[feature, 'global_dominant_share_pct']:.2f}%; "
            f"uplift "
            f"{lei_lookup_5_1_reviewed.loc[feature, 'lei_uplift_vs_global_pp']:+.2f} pp."
        )
        for feature in GROUP_5_1_COLUMNS
    })
)

screening_decision_5_1["business_reason"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": "Distinct conditional preapproval-request concept.",
        "submission_of_application": "Application-submission process.",
        "initially_payable_to_institution": "Initial institutional relationship.",
        "conforming_loan_limit": "Reported conforming-loan-limit status.",
    })
)
screening_decision_5_1["protected_group_sensitivity"] = (
    "Descriptive coverage only; not a decision rule or predictor."
)
screening_decision_5_1["screening_status"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": "keep_after_data_governance_review",
        "submission_of_application": "exclude_from_further_review",
        "initially_payable_to_institution": "exclude_from_further_review",
        "conforming_loan_limit": "exclude_from_further_review",
    })
)
screening_decision_5_1["decision"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": "keep",
        "submission_of_application": "exclude",
        "initially_payable_to_institution": "exclude",
        "conforming_loan_limit": "exclude",
    })
)
screening_decision_5_1["decision_reason"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": (
            "Kept as a distinct pre-decision application concept with valid "
            "conditional applicability and low LEI uplift."
        ),
        "submission_of_application": (
            "Excluded with the application-process block because it mainly "
            "describes submission and organization process."
        ),
        "initially_payable_to_institution": (
            "Excluded with the application-process block because it mainly "
            "describes institutional process."
        ),
        "conforming_loan_limit": (
            "Excluded because the reported loan-amount basis differs by action "
            "outcome; retain only in source/audit."
        ),
    })
)
screening_decision_5_1["gsv_player"] = (
    screening_decision_5_1["feature_or_block"].map({
        "preapproval": "preapproval",
        "submission_of_application": "not_applicable",
        "initially_payable_to_institution": "not_applicable",
        "conforming_loan_limit": "not_applicable",
    })
)
screening_decision_5_1["validation_evidence"] = (
    "not_run_not_required_for_data_governance_scope"
)
screening_decision_5_1["stability_evidence"] = (
    screening_decision_5_1["decision"].map({
        "keep": "data_level_evidence_reviewed_no_model_resampling",
        "exclude": "not_applicable",
    })
)
conforming_row_mask_5_1 = screening_decision_5_1["feature_or_block"].eq(
    "conforming_loan_limit"
)
timing_resolution_mask_5_1 = (
    screening_decision_5_1["timing_status"].eq(
        "available_before_decision_by_definition"
    )
    | (
        screening_decision_5_1["timing_status"].eq(
            "not_consistently_predecision_safe"
        )
        & screening_decision_5_1["decision"].eq("exclude")
    )
)
section_5_1_gate = pd.Series({
    "schema_gate": bool(validation_5_1_reviewed[
        "schema_map_matches_observed_tokens"
    ]),
    "informative_redundancy_gate": pd.notna(
        informative_pair_cramers_v_5_1
    ),
    "conditional_applicability_gate": (
        requested_home_purchase_n_5_1 == requested_preapproval_n_5_1
    ),
    "lei_diagnostic_gate": lei_proxy_enhanced_5_1[
        "lei_uplift_vs_global_pp"
    ].notna().all(),
    "screening_record_gate": screening_decision_5_1[
        "screening_status"
    ].notna().all(),
    "timing_gate": timing_resolution_mask_5_1.all(),
    "decision_record_gate": (
        screening_decision_5_1["decision"].isin(["keep", "exclude"]).all()
        and screening_decision_5_1["gsv_player"].notna().all()
    ),
})
section_5_1_gate["ready_to_close_5_1"] = bool(section_5_1_gate.all())
section_5_1_gate["ready_to_start_5_2"] = bool(
    section_5_1_gate["ready_to_close_5_1"]
)
assert screening_decision_5_1["decision"].isin(["keep", "exclude"]).all()
assert screening_decision_5_1["gsv_player"].notna().all()
assert section_5_1_gate["ready_to_start_5_2"]

display(screening_decision_5_1)
display(section_5_1_gate)

,feature_or_block,timing_status,timing_evidence,quality_evidence,conditional_applicability,redundancy_evidence,organization_proxy_evidence,business_reason,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence
0,preapproval,available_before_decision_by_definition,2024 HMDA FIG 4(a)(4): whether the application...,100.00% valid; 100.00% informative; 0 invalid ...,All 9516 requested-preapproval rows are home p...,Conditional on home-purchase scope; not a dupl...,LEI row-weighted dominance 97.34%; global domi...,Distinct conditional preapproval-request concept.,Descriptive coverage only; not a decision rule...,keep_after_data_governance_review,keep,Kept as a distinct pre-decision application co...,preapproval,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
1,submission_of_application,available_before_decision_by_definition,2024 HMDA FIG 4(a)(33): whether the applicant ...,100.00% valid; 97.18% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with initially_payable: full V=0.7...,LEI row-weighted dominance 96.30%; global domi...,Application-submission process.,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded with the application-process block be...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
2,initially_payable_to_institution,available_before_decision_by_definition,"2024 HMDA FIG 4(a)(33): for an application, wh...",100.00% valid; 95.29% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with submission: full V=0.7178; in...,LEI row-weighted dominance 96.57%; global domi...,Initial institutional relationship.,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded with the application-process block be...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
3,conforming_loan_limit,not_consistently_predecision_safe,CFPB Regulation C timing: reported loan amount...,100.00% valid; 98.92% informative; 0 invalid c...,U and NA retained as separate valid special st...,Public-derived from outcome-dependent reported...,LEI row-weighted dominance 95.93%; global domi...,Reported conforming-loan-limit status.,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded because the reported loan-amount basi...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable


schema_gate                       True
informative_redundancy_gate       True
conditional_applicability_gate    True
lei_diagnostic_gate               True
screening_record_gate             True
timing_gate                       True
decision_record_gate              True
ready_to_close_5_1                True
ready_to_start_5_2                True
dtype: bool

### 5.1.2. Kết luận screening

**Observed facts**

- `NA` của `conforming_loan_limit` là valid special state (*Not Applicable*), không phải invalid code. Bốn feature có valid-code coverage 100%.
- `submission_of_application` và `initially_payable_to_institution` có Cramér's V bằng 0,7178 trên full cohort và 0,1735 trên informative-only population.
- Toàn bộ 9.516 hồ sơ requested-preapproval thuộc home purchase; đây là conditional applicability, không phải duplication.
- LEI chỉ là diagnostic metadata và không được đưa vào `X`.

**Final screening decisions**

- `preapproval`: **keep**, `gsv_player=preapproval`.
- `submission_of_application` và `initially_payable_to_institution`: **exclude** như application-process block, `gsv_player=not_applicable`.
- `conforming_loan_limit`: **exclude** do không nhất quán pre-decision-safe, `gsv_player=not_applicable`.

Phần 5.1 đóng bằng data-governance evidence; không sử dụng model performance, feature importance hoặc model validation.


## 5.2. Cấu trúc trả nợ đặc biệt

Screening này đánh giá bốn repayment flags về schema, timing, coverage, overlap, LEI guardrail và protected-metadata sensitivity. Không fit model, không mở test set và không quyết định final keep/exclude. Theo 2024 HMDA FIG 4(a)(27), code `1`, `2` và `1111` được giữ riêng; `1111` là reporting exemption, không phải "No".


In [17]:
GROUP_5_2_COLUMNS = [
    "negative_amortization",
    "interest_only_payment",
    "balloon_payment",
    "other_nonamortizing_features",
]
RACE_SLOT_COLUMNS_5_2 = [f"applicant_race-{slot}" for slot in range(1, 6)]
REFERENCE_5_2_COLUMNS = ["lei", "applicant_sex"] + RACE_SLOT_COLUMNS_5_2
required_columns_5_2 = GROUP_5_2_COLUMNS + REFERENCE_5_2_COLUMNS
missing_columns_5_2 = [
    column for column in required_columns_5_2
    if column not in dataset_b.columns
]
assert not missing_columns_5_2, f"Missing columns: {missing_columns_5_2}"
group_5_2_data = dataset_b.loc[:, required_columns_5_2].copy()

EXPECTED_CODE_LABELS_5_2 = {
    "negative_amortization": {
        "1": "Negative amortization",
        "2": "No negative amortization",
        "1111": "Exempt",
    },
    "interest_only_payment": {
        "1": "Interest-only payments",
        "2": "No interest-only payments",
        "1111": "Exempt",
    },
    "balloon_payment": {
        "1": "Balloon payment",
        "2": "No balloon payment",
        "1111": "Exempt",
    },
    "other_nonamortizing_features": {
        "1": "Other non-fully amortizing features",
        "2": "No other non-amortizing features",
        "1111": "Exempt",
    },
}
SPECIAL_CODES_5_2 = {feature: {"1111"} for feature in GROUP_5_2_COLUMNS}
INFORMATIVE_CODES_5_2 = {"1", "2"}

def audit_feature_codes_5_2(data, feature):
    rows = []
    for raw_code, count in data[feature].value_counts(dropna=False).items():
        if pd.isna(raw_code) or str(raw_code).strip() == "":
            status, description = "blank_or_missing", "Blank or missing"
        elif raw_code not in EXPECTED_CODE_LABELS_5_2[feature]:
            status, description = "invalid_code", "Unrecognized code"
        elif raw_code in SPECIAL_CODES_5_2[feature]:
            status, description = "valid_special", EXPECTED_CODE_LABELS_5_2[feature][raw_code]
        else:
            status, description = "valid_informative", EXPECTED_CODE_LABELS_5_2[feature][raw_code]
        rows.append({
            "feature": feature,
            "raw_code": raw_code,
            "description": description,
            "status": status,
            "n": int(count),
            "share_pct": round(count / len(data) * 100, 4),
        })
    return pd.DataFrame(rows)

audit_5_2 = pd.concat(
    [audit_feature_codes_5_2(group_5_2_data, feature) for feature in GROUP_5_2_COLUMNS],
    ignore_index=True,
)
quality_rows_5_2 = []
for feature in GROUP_5_2_COLUMNS:
    counts = (
        audit_5_2.loc[audit_5_2["feature"].eq(feature)]
        .groupby("status")["n"]
        .sum()
    )
    informative_n = int(counts.get("valid_informative", 0))
    special_n = int(counts.get("valid_special", 0))
    positive_n = int(group_5_2_data[feature].eq("1").sum())
    quality_rows_5_2.append({
        "feature": feature,
        "row_count": len(group_5_2_data),
        "valid_n": informative_n + special_n,
        "valid_pct": round((informative_n + special_n) / len(group_5_2_data) * 100, 4),
        "informative_n": informative_n,
        "informative_pct": round(informative_n / len(group_5_2_data) * 100, 4),
        "positive_n": positive_n,
        "positive_pct": round(positive_n / len(group_5_2_data) * 100, 4),
        "special_n": special_n,
        "special_pct": round(special_n / len(group_5_2_data) * 100, 4),
        "blank_or_missing_n": int(counts.get("blank_or_missing", 0)),
        "invalid_code_n": int(counts.get("invalid_code", 0)),
    })
quality_5_2 = pd.DataFrame(quality_rows_5_2)
invalid_codes_5_2 = audit_5_2.loc[audit_5_2["status"].eq("invalid_code")].copy()
quality_validation_5_2 = pd.Series({
    "row_count_reconciled": quality_5_2["row_count"].eq(len(dataset_b)).all(),
    "audit_counts_reconciled": audit_5_2.groupby("feature")["n"].sum().eq(len(dataset_b)).all(),
    "schema_map_matches_observed_tokens": invalid_codes_5_2.empty,
})
assert quality_validation_5_2.all()
display(quality_5_2)
display(audit_5_2)
display(quality_validation_5_2)


,feature,row_count,valid_n,valid_pct,informative_n,informative_pct,positive_n,positive_pct,special_n,special_pct,blank_or_missing_n,invalid_code_n
0,negative_amortization,293301,293301,100.0,284657,97.0529,401,0.1367,8644,2.9471,0,0
1,interest_only_payment,293301,293301,100.0,284657,97.0529,56571,19.2877,8644,2.9471,0,0
2,balloon_payment,293301,293301,100.0,284657,97.0529,6965,2.3747,8644,2.9471,0,0
3,other_nonamortizing_features,293301,293301,100.0,284657,97.0529,411,0.1401,8644,2.9471,0,0


,feature,raw_code,description,status,n,share_pct
0,negative_amortization,2,No negative amortization,valid_informative,284256,96.9161
1,negative_amortization,1111,Exempt,valid_special,8644,2.9471
2,negative_amortization,1,Negative amortization,valid_informative,401,0.1367
3,interest_only_payment,2,No interest-only payments,valid_informative,228086,77.7652
4,interest_only_payment,1,Interest-only payments,valid_informative,56571,19.2877
5,interest_only_payment,1111,Exempt,valid_special,8644,2.9471
6,balloon_payment,2,No balloon payment,valid_informative,277692,94.6782
7,balloon_payment,1111,Exempt,valid_special,8644,2.9471
8,balloon_payment,1,Balloon payment,valid_informative,6965,2.3747
9,other_nonamortizing_features,2,No other non-amortizing features,valid_informative,284246,96.9127


row_count_reconciled                  True
audit_counts_reconciled               True
schema_map_matches_observed_tokens    True
dtype: bool

In [18]:
def cramers_v_5_2(data, left, right):
    observed = pd.crosstab(data[left], data[right], dropna=False).to_numpy(dtype=float)
    n = observed.sum()
    if n == 0 or min(observed.shape) < 2:
        return np.nan
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / n
    chi_square = np.divide(
        (observed - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected > 0,
    ).sum()
    return float(
        np.sqrt(
            (chi_square / n)
            / min(observed.shape[0] - 1, observed.shape[1] - 1)
        )
    )

informative_mask_5_2 = group_5_2_data[GROUP_5_2_COLUMNS].isin(
    INFORMATIVE_CODES_5_2
).all(axis=1)
informative_data_5_2 = group_5_2_data.loc[informative_mask_5_2].copy()
association_rows_5_2 = []
for left, right in combinations(GROUP_5_2_COLUMNS, 2):
    for population, data in [
        ("full_cohort", group_5_2_data),
        ("informative_only", informative_data_5_2),
    ]:
        association_rows_5_2.append({
            "left_feature": left,
            "right_feature": right,
            "population": population,
            "n": len(data),
            "cramers_v": round(cramers_v_5_2(data, left, right), 4),
        })
association_5_2 = pd.DataFrame(association_rows_5_2)
association_by_feature_5_2 = pd.concat([
    association_5_2.rename(columns={"left_feature": "feature"})[
        ["feature", "population", "cramers_v"]
    ],
    association_5_2.rename(columns={"right_feature": "feature"})[
        ["feature", "population", "cramers_v"]
    ],
]).groupby(["feature", "population"])["cramers_v"].max().unstack("population")

repayment_patterns_5_2 = (
    group_5_2_data.groupby(GROUP_5_2_COLUMNS, dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n", ascending=False)
    .reset_index(drop=True)
)
repayment_patterns_5_2["share_pct"] = (
    repayment_patterns_5_2["n"] / len(group_5_2_data) * 100
).round(4)
any_special_repayment_5_2 = group_5_2_data[GROUP_5_2_COLUMNS].eq("1").any(axis=1)
block_prevalence_5_2 = pd.Series({
    "full_cohort_n": len(group_5_2_data),
    "informative_all_four_n": int(informative_mask_5_2.sum()),
    "any_special_repayment_n": int(any_special_repayment_5_2.sum()),
    "any_special_repayment_pct": round(any_special_repayment_5_2.mean() * 100, 4),
})
assert repayment_patterns_5_2["n"].sum() == len(group_5_2_data)
assert len(informative_data_5_2) > 0
display(block_prevalence_5_2)
display(association_5_2)
display(repayment_patterns_5_2)


full_cohort_n                293301.0000
informative_all_four_n       284657.0000
any_special_repayment_n       59655.0000
any_special_repayment_pct        20.3392
dtype: float64

,left_feature,right_feature,population,n,cramers_v
0,negative_amortization,interest_only_payment,full_cohort,293301,0.7072
1,negative_amortization,interest_only_payment,informative_only,284657,0.0168
2,negative_amortization,balloon_payment,full_cohort,293301,0.7071
3,negative_amortization,balloon_payment,informative_only,284657,0.0017
4,negative_amortization,other_nonamortizing_features,full_cohort,293301,0.7071
5,negative_amortization,other_nonamortizing_features,informative_only,284657,0.0010
6,interest_only_payment,balloon_payment,full_cohort,293301,0.7173
7,interest_only_payment,balloon_payment,informative_only,284657,0.1701
8,interest_only_payment,other_nonamortizing_features,full_cohort,293301,0.7075
9,interest_only_payment,other_nonamortizing_features,informative_only,284657,0.0337


,negative_amortization,interest_only_payment,balloon_payment,other_nonamortizing_features,n,share_pct
0,2,2,2,2,225002,76.7137
1,2,1,2,2,51986,17.7245
2,1111,1111,1111,1111,8644,2.9471
3,2,1,1,2,4351,1.4835
4,2,2,1,2,2507,0.8548
5,1,2,2,2,393,0.1340
6,2,1,2,1,214,0.0730
7,2,2,2,1,96,0.0327
8,2,2,1,1,88,0.0300
9,2,1,1,1,12,0.0041


In [19]:
valid_lei_mask_5_2 = (
    group_5_2_data["lei"].notna()
    & group_5_2_data["lei"].str.strip().ne("")
)
lei_data_5_2 = group_5_2_data.loc[
    valid_lei_mask_5_2,
    ["lei"] + GROUP_5_2_COLUMNS,
].copy()
lei_rows_5_2 = []
for feature in GROUP_5_2_COLUMNS:
    counts = (
        lei_data_5_2.groupby(["lei", feature], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )
    counts["lei_total_n"] = counts.groupby("lei")["n"].transform("sum")
    per_lei = (
        counts.assign(within_lei_share=counts["n"] / counts["lei_total_n"])
        .groupby("lei")
        .agg(
            row_count=("n", "sum"),
            dominant_share=("within_lei_share", "max"),
        )
        .reset_index()
    )
    positive_by_lei = (
        lei_data_5_2.assign(is_positive=lei_data_5_2[feature].eq("1"))
        .groupby("lei")["is_positive"]
        .sum()
    )
    positive_n = int(positive_by_lei.sum())
    positive_lei_n = int(positive_by_lei.gt(0).sum())
    global_dominance = (
        lei_data_5_2[feature].value_counts(dropna=False).iloc[0]
        / len(lei_data_5_2)
        * 100
    )
    weighted_dominance = (
        np.average(per_lei["dominant_share"], weights=per_lei["row_count"])
        * 100
    )
    lei_rows_5_2.append({
        "feature": feature,
        "lei_n": per_lei["lei"].nunique(),
        "global_dominant_share_pct": round(global_dominance, 4),
        "row_weighted_dominant_share_pct": round(weighted_dominance, 4),
        "lei_uplift_vs_global_pp": round(weighted_dominance - global_dominance, 4),
        "positive_lei_n": positive_lei_n,
        "positive_top_1_lei_share_pct": (
            0.0 if positive_n == 0 else round(positive_by_lei.nlargest(1).sum() / positive_n * 100, 4)
        ),
        "positive_top_10_lei_share_pct": (
            0.0
            if positive_n == 0
            else round(positive_by_lei.nlargest(10).sum() / positive_n * 100, 4)
        ),
    })
lei_proxy_5_2 = pd.DataFrame(lei_rows_5_2)

def canonical_race_token_set_5_2(row):
    tokens = sorted({str(value).strip() for value in row if str(value).strip()})
    return "|".join(tokens) if tokens else "blank"

protected_data_5_2 = group_5_2_data[
    ["applicant_sex"] + RACE_SLOT_COLUMNS_5_2 + GROUP_5_2_COLUMNS
].copy()
protected_data_5_2["applicant_race_token_set"] = protected_data_5_2[
    RACE_SLOT_COLUMNS_5_2
].apply(canonical_race_token_set_5_2, axis=1)
protected_rows_5_2 = []
for protected_group in ["applicant_sex", "applicant_race_token_set"]:
    for feature in GROUP_5_2_COLUMNS:
        summary = (
            protected_data_5_2.groupby(protected_group, dropna=False)[feature]
            .agg(
                n="size",
                positive_n=lambda values: values.eq("1").sum(),
                special_n=lambda values: values.eq("1111").sum(),
            )
            .reset_index()
        )
        summary["protected_group"] = protected_group
        summary["feature"] = feature
        summary["positive_pct"] = (summary["positive_n"] / summary["n"] * 100).round(4)
        summary["special_pct"] = (summary["special_n"] / summary["n"] * 100).round(4)
        protected_rows_5_2.append(summary)
protected_sensitivity_5_2 = pd.concat(protected_rows_5_2, ignore_index=True)
assert lei_proxy_5_2["lei_uplift_vs_global_pp"].notna().all()
assert not protected_sensitivity_5_2.empty
display(lei_proxy_5_2)
display(protected_sensitivity_5_2.sort_values("n", ascending=False).head(40))

quality_lookup_5_2 = quality_5_2.set_index("feature")
lei_lookup_5_2 = lei_proxy_5_2.set_index("feature")
screening_decision_5_2 = pd.DataFrame({"feature_or_block": GROUP_5_2_COLUMNS})
screening_decision_5_2["timing_status"] = "available_before_decision_by_definition"
screening_decision_5_2["timing_evidence"] = (
    "2024 HMDA FIG 4(a)(27): contractual terms include, or would have included, "
    "the repayment feature for a covered loan or application."
)
screening_decision_5_2["quality_evidence"] = screening_decision_5_2[
    "feature_or_block"
].map({
    feature: (
        f"{quality_lookup_5_2.loc[feature, 'valid_pct']:.2f}% valid; "
        f"{quality_lookup_5_2.loc[feature, 'informative_pct']:.2f}% informative; "
        f"{quality_lookup_5_2.loc[feature, 'positive_pct']:.2f}% code-1; "
        f"{int(quality_lookup_5_2.loc[feature, 'invalid_code_n'])} invalid codes."
    )
    for feature in GROUP_5_2_COLUMNS
})
screening_decision_5_2["conditional_applicability"] = (
    "All-cohort reporting field; code 1111 retained as reporting exemption, not imputed as No."
)
screening_decision_5_2["redundancy_evidence"] = screening_decision_5_2[
    "feature_or_block"
].map({
    feature: (
        f"Maximum pairwise V: full={association_by_feature_5_2.loc[feature, 'full_cohort']:.4f}; "
        f"informative-only={association_by_feature_5_2.loc[feature, 'informative_only']:.4f}."
    )
    for feature in GROUP_5_2_COLUMNS
})
screening_decision_5_2["organization_proxy_evidence"] = screening_decision_5_2[
    "feature_or_block"
].map({
    feature: (
        f"LEI row-weighted dominance {lei_lookup_5_2.loc[feature, 'row_weighted_dominant_share_pct']:.2f}%; "
        f"global {lei_lookup_5_2.loc[feature, 'global_dominant_share_pct']:.2f}%; "
        f"uplift {lei_lookup_5_2.loc[feature, 'lei_uplift_vs_global_pp']:+.2f} pp; "
        f"code-1 occurs in {int(lei_lookup_5_2.loc[feature, 'positive_lei_n'])} LEIs; "
        f"top-1/top-10 LEIs hold {lei_lookup_5_2.loc[feature, 'positive_top_1_lei_share_pct']:.2f}%/"
        f"{lei_lookup_5_2.loc[feature, 'positive_top_10_lei_share_pct']:.2f}% of code-1 rows."
    )
    for feature in GROUP_5_2_COLUMNS
})
screening_decision_5_2["business_reason"] = (
    screening_decision_5_2["feature_or_block"].map({
        "negative_amortization": "Contractual negative-amortization term.",
        "interest_only_payment": "Contractual interest-only payment term.",
        "balloon_payment": "Contractual balloon-payment term.",
        "other_nonamortizing_features": "Other non-amortizing contractual term.",
    })
)
screening_decision_5_2["protected_group_sensitivity"] = (
    "Descriptive coverage only; not a decision rule or predictor."
)
screening_decision_5_2["screening_status"] = (
    screening_decision_5_2["feature_or_block"].map({
        "negative_amortization": "exclude_from_further_review",
        "interest_only_payment": "keep_after_data_governance_review",
        "balloon_payment": "keep_after_data_governance_review",
        "other_nonamortizing_features": "exclude_from_further_review",
    })
)
screening_decision_5_2["decision"] = (
    screening_decision_5_2["feature_or_block"].map({
        "negative_amortization": "exclude",
        "interest_only_payment": "keep",
        "balloon_payment": "keep",
        "other_nonamortizing_features": "exclude",
    })
)
screening_decision_5_2["decision_reason"] = (
    screening_decision_5_2["feature_or_block"].map({
        "negative_amortization": (
            "Excluded for extreme rarity and concentration of code-1 "
            "observations in very few LEIs."
        ),
        "interest_only_payment": (
            "Kept with balloon_payment as a distinct contractual "
            "special-repayment block."
        ),
        "balloon_payment": (
            "Kept with interest_only_payment as a distinct contractual "
            "special-repayment block."
        ),
        "other_nonamortizing_features": (
            "Excluded for extreme rarity and organizational concentration."
        ),
    })
)
screening_decision_5_2["gsv_player"] = (
    screening_decision_5_2["decision"].map({
        "keep": "special_repayment_structure",
        "exclude": "not_applicable",
    })
)
screening_decision_5_2["validation_evidence"] = (
    "not_run_not_required_for_data_governance_scope"
)
screening_decision_5_2["stability_evidence"] = (
    screening_decision_5_2["decision"].map({
        "keep": "data_level_evidence_reviewed_no_model_resampling",
        "exclude": "not_applicable",
    })
)
excluded_proxy_mask_5_2 = screening_decision_5_2["decision"].eq("exclude")
section_5_2_gate = pd.Series({
    "prior_5_1_gate": bool(section_5_1_gate["ready_to_start_5_2"]),
    "schema_gate": bool(quality_validation_5_2[
        "schema_map_matches_observed_tokens"
    ]),
    "timing_gate": screening_decision_5_2["timing_status"].eq(
        "available_before_decision_by_definition"
    ).all(),
    "informative_overlap_gate": association_5_2.loc[
        association_5_2["population"].eq("informative_only"), "cramers_v"
    ].notna().all(),
    "lei_diagnostic_gate": lei_proxy_5_2[
        "lei_uplift_vs_global_pp"
    ].notna().all(),
    "protected_sensitivity_gate": not protected_sensitivity_5_2.empty,
    "screening_record_gate": screening_decision_5_2[
        "screening_status"
    ].notna().all(),
    "decision_record_gate": (
        screening_decision_5_2["decision"].isin(["keep", "exclude"]).all()
        and screening_decision_5_2["gsv_player"].notna().all()
    ),
})
section_5_2_gate["ready_to_close_5_2"] = bool(section_5_2_gate.all())
section_5_2_gate["ready_to_start_5_3"] = bool(
    section_5_2_gate["ready_to_close_5_2"]
)
assert screening_decision_5_2["decision"].isin(["keep", "exclude"]).all()
assert screening_decision_5_2["gsv_player"].notna().all()
assert section_5_2_gate["ready_to_close_5_2"]

display(screening_decision_5_2)
display(section_5_2_gate)


,feature,lei_n,global_dominant_share_pct,row_weighted_dominant_share_pct,lei_uplift_vs_global_pp,positive_lei_n,positive_top_1_lei_share_pct,positive_top_10_lei_share_pct
0,negative_amortization,715,96.9161,99.9489,3.0327,4,97.7556,100.0000
1,interest_only_payment,715,77.7652,89.3730,11.6079,278,17.8077,70.2127
2,balloon_payment,715,94.6782,98.3392,3.6611,179,19.6267,57.3439
3,other_nonamortizing_features,715,96.9127,99.8285,2.9158,18,53.2847,97.3236


,applicant_sex,n,positive_n,special_n,protected_group,feature,positive_pct,special_pct,applicant_race_token_set
920,<NA>,187507,181,5990,applicant_race_token_set,other_nonamortizing_features,0.0965,3.1945,5
468,<NA>,187507,36050,5990,applicant_race_token_set,interest_only_payment,19.2259,3.1945,5
694,<NA>,187507,1031,5990,applicant_race_token_set,balloon_payment,0.5498,3.1945,5
242,<NA>,187507,248,5990,applicant_race_token_set,negative_amortization,0.1323,3.1945,5
0,1,164764,192,4659,applicant_sex,negative_amortization,0.1165,2.8277,NaN
10,1,164764,1078,4659,applicant_sex,balloon_payment,0.6543,2.8277,NaN
5,1,164764,31973,4659,applicant_sex,interest_only_payment,19.4053,2.8277,NaN
15,1,164764,174,4659,applicant_sex,other_nonamortizing_features,0.1056,2.8277,NaN
16,2,102299,79,2905,applicant_sex,other_nonamortizing_features,0.0772,2.8397,NaN
11,2,102299,410,2905,applicant_sex,balloon_payment,0.4008,2.8397,NaN


,feature_or_block,timing_status,timing_evidence,quality_evidence,conditional_applicability,redundancy_evidence,organization_proxy_evidence,business_reason,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence
0,negative_amortization,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 0.14% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7072; informative-o...,LEI row-weighted dominance 99.95%; global 96.9...,Contractual negative-amortization term.,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded for extreme rarity and concentration ...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
1,interest_only_payment,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 19.29% code...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7173; informative-o...,LEI row-weighted dominance 89.37%; global 77.7...,Contractual interest-only payment term.,Descriptive coverage only; not a decision rule...,keep_after_data_governance_review,keep,Kept with balloon_payment as a distinct contra...,special_repayment_structure,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
2,balloon_payment,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 2.37% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7173; informative-o...,LEI row-weighted dominance 98.34%; global 94.6...,Contractual balloon-payment term.,Descriptive coverage only; not a decision rule...,keep_after_data_governance_review,keep,Kept with interest_only_payment as a distinct ...,special_repayment_structure,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
3,other_nonamortizing_features,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 0.14% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7081; informative-o...,LEI row-weighted dominance 99.83%; global 96.9...,Other non-amortizing contractual term.,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded for extreme rarity and organizational...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable


prior_5_1_gate                True
schema_gate                   True
timing_gate                   True
informative_overlap_gate      True
lei_diagnostic_gate           True
protected_sensitivity_gate    True
screening_record_gate         True
decision_record_gate          True
ready_to_close_5_2            True
ready_to_start_5_3            True
dtype: bool

### 5.2.1. Kết luận screening

Bốn field có valid-code coverage 100%; `1111` được giữ là reporting exemption, không bị gộp với “No”. Informative-only association lớn nhất là `interest_only_payment` × `balloon_payment` với Cramér's V = 0,1701. LEI và protected metadata chỉ là diagnostic guardrails.

**Final screening decisions**

- `negative_amortization`: **exclude**, `gsv_player=not_applicable`.
- `other_nonamortizing_features`: **exclude**, `gsv_player=not_applicable`.
- `interest_only_payment` và `balloon_payment`: **keep** như một joint contractual block, cùng `gsv_player=special_repayment_structure`.

Phần 5.2 đóng bằng data-governance evidence; không sử dụng model performance, feature importance hoặc model validation.


### 5.3. Manufactured home và multifamily — audited screening

**Phạm vi:** `manufactured_home_secured_property_type`, `manufactured_home_land_property_interest`, và `multifamily_affordable_units` trên cohort `dataset_b` (`action_taken ∈ {1, 2, 3}`).

**Ranh giới:** special states được bảo toàn; không impute `Not Applicable` thành `0`; `lei`, `applicant_sex` và năm race slots chỉ là diagnostic metadata, không vào `X`. FIG xác nhận domain báo cáo. Prediction-time lineage được tách riêng và không tự suy ra từ domain validation.

In [20]:
# 5.3.1 Schema and numeric audit — raw tokens remain distinct
from math import isfinite

GROUP_5_3_COLUMNS = [
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest",
    "multifamily_affordable_units",
]
RACE_SLOT_COLUMNS_5_3 = [f"applicant_race-{slot}" for slot in range(1, 6)]
SCHEMA_5_3 = {
    "manufactured_home_secured_property_type": {
        "informative": {"1", "2"}, "not_applicable": {"3"}, "exempt": {"1111", "EXEMPT"},
        "fig_reference": "2024 HMDA FIG field 89",
    },
    "manufactured_home_land_property_interest": {
        "informative": {"1", "2", "3", "4"}, "not_applicable": {"5"}, "exempt": {"1111", "EXEMPT"},
        "fig_reference": "2024 HMDA FIG field 90",
    },
    "multifamily_affordable_units": {
        "informative": "nonnegative_integer", "not_applicable": {"NA"}, "exempt": {"1111", "EXEMPT"},
        "fig_reference": "2024 HMDA FIG field 92",
    },
}

SPECIAL_BLANK_5_3 = {"", "NAN", "NONE", "<NA>"}
def normalized_token_5_3(value):
    return str(value).strip().upper()

def status_5_3(value, feature):
    token = normalized_token_5_3(value)
    if pd.isna(value) or token in SPECIAL_BLANK_5_3:
        return "Blank"
    rule = SCHEMA_5_3[feature]
    if token in rule["exempt"]:
        return "Exempt"
    if token in rule["not_applicable"]:
        return "Not Applicable"
    if feature != "multifamily_affordable_units":
        return "Informative" if token in rule["informative"] else "Invalid"
    try:
        numeric_value = float(token)
    except ValueError:
        return "Invalid"
    return "Informative" if isfinite(numeric_value) and numeric_value >= 0 and numeric_value.is_integer() else "Invalid"

analysis_5_3 = dataset_b.copy()
for feature in GROUP_5_3_COLUMNS:
    analysis_5_3[f"{feature}_status"] = analysis_5_3[feature].map(lambda value: status_5_3(value, feature))

quality_rows_5_3 = []
for feature in GROUP_5_3_COLUMNS:
    counts = analysis_5_3[f"{feature}_status"].value_counts()
    quality_rows_5_3.append({
        "feature": feature,
        "fig_reference": SCHEMA_5_3[feature]["fig_reference"],
        "row_count": len(analysis_5_3),
        "informative_n": int(counts.get("Informative", 0)),
        "not_applicable_n": int(counts.get("Not Applicable", 0)),
        "exempt_n": int(counts.get("Exempt", 0)),
        "blank_n": int(counts.get("Blank", 0)),
        "invalid_n": int(counts.get("Invalid", 0)),
    })
quality_5_3 = pd.DataFrame(quality_rows_5_3)
quality_5_3["reconciled_n"] = quality_5_3[["informative_n", "not_applicable_n", "exempt_n", "blank_n", "invalid_n"]].sum(axis=1)
quality_validation_5_3 = pd.Series({
    "cohort_is_dataset_b": len(analysis_5_3) == len(dataset_b),
    "all_counts_reconciled": quality_5_3["reconciled_n"].eq(len(dataset_b)).all(),
    "no_invalid_tokens": quality_5_3["invalid_n"].eq(0).all(),
    "no_blank_tokens": quality_5_3["blank_n"].eq(0).all(),
})
assert quality_validation_5_3[["cohort_is_dataset_b", "all_counts_reconciled"]].all()
display(quality_5_3)
display(quality_validation_5_3)

,feature,fig_reference,row_count,informative_n,not_applicable_n,exempt_n,blank_n,invalid_n,reconciled_n
0,manufactured_home_secured_property_type,2024 HMDA FIG field 89,293301,6262,278463,8576,0,0,293301
1,manufactured_home_land_property_interest,2024 HMDA FIG field 90,293301,6258,278424,8619,0,0,293301
2,multifamily_affordable_units,2024 HMDA FIG field 92,293301,2434,282219,8648,0,0,293301


cohort_is_dataset_b      True
all_counts_reconciled    True
no_invalid_tokens        True
no_blank_tokens          True
dtype: bool

In [21]:
# 5.3.2 Applicability audit — context first, then observed status
MULTIFAMILY_TOTAL_UNIT_TOKENS_5_3 = {"5-24", "25-49", "50-99", "100-149", ">149"}
analysis_5_3["is_manufactured_context"] = analysis_5_3["construction_method"].astype(str).str.strip().eq("2")
analysis_5_3["is_multifamily_context"] = analysis_5_3["total_units"].astype(str).str.strip().isin(MULTIFAMILY_TOTAL_UNIT_TOKENS_5_3)

applicability_specs_5_3 = [
    ("manufactured_home_secured_property_type", "is_manufactured_context"),
    ("manufactured_home_land_property_interest", "is_manufactured_context"),
    ("multifamily_affordable_units", "is_multifamily_context"),
]
applicability_rows_5_3 = []
for feature, context_column in applicability_specs_5_3:
    status_column = f"{feature}_status"
    in_context = analysis_5_3[context_column]
    statuses = analysis_5_3[status_column]
    applicability_rows_5_3.append({
        "feature": feature,
        "context_column": context_column,
        "applicable_context_n": int(in_context.sum()),
        "outside_context_n": int((~in_context).sum()),
        "outside_context_informative_n": int((~in_context & statuses.eq("Informative")).sum()),
        "in_context_not_applicable_n": int((in_context & statuses.eq("Not Applicable")).sum()),
        "in_context_exempt_n": int((in_context & statuses.eq("Exempt")).sum()),
        "in_context_blank_n": int((in_context & statuses.eq("Blank")).sum()),
    })
applicability_5_3 = pd.DataFrame(applicability_rows_5_3)

manufactured_cross_tabs_5_3 = {
    feature: pd.crosstab(
        analysis_5_3["construction_method"],
        analysis_5_3[f"{feature}_status"],
        margins=True,
    )
    for feature in ["manufactured_home_secured_property_type", "manufactured_home_land_property_interest"]
}
multifamily_cross_tab_5_3 = pd.crosstab(
    analysis_5_3["total_units"],
    analysis_5_3["multifamily_affordable_units_status"],
    margins=True,
)
applicability_validation_5_3 = pd.Series({
    "no_outside_context_informative_values": applicability_5_3["outside_context_informative_n"].eq(0).all(),
    "context_exceptions_recorded": applicability_5_3[["in_context_not_applicable_n", "in_context_exempt_n", "in_context_blank_n"]].notna().all().all(),
})
display(applicability_5_3)
display(applicability_validation_5_3)
for feature, table in manufactured_cross_tabs_5_3.items():
    print(feature)
    display(table)
display(multifamily_cross_tab_5_3)

,feature,context_column,applicable_context_n,outside_context_n,outside_context_informative_n,in_context_not_applicable_n,in_context_exempt_n,in_context_blank_n
0,manufactured_home_secured_property_type,is_manufactured_context,6815,286486,0,26,527,0
1,manufactured_home_land_property_interest,is_manufactured_context,6815,286486,0,28,529,0
2,multifamily_affordable_units,is_multifamily_context,3107,290194,0,163,510,0


no_outside_context_informative_values    True
context_exceptions_recorded              True
dtype: bool

manufactured_home_secured_property_type


manufactured_home_secured_property_type_status,Exempt,Informative,Not Applicable,All
construction_method,,,,
1,8049,0,278437,286486
2,527,6262,26,6815
All,8576,6262,278463,293301


manufactured_home_land_property_interest


manufactured_home_land_property_interest_status,Exempt,Informative,Not Applicable,All
construction_method,,,,
1,8090,0,278396,286486
2,529,6258,28,6815
All,8619,6258,278424,293301


multifamily_affordable_units_status,Exempt,Informative,Not Applicable,All
total_units,,,,
1,7152,0,252632,259784
100-149,17,88,3,108
2,723,0,23182,23905
25-49,57,342,6,405
3,148,0,4386,4534
4,115,0,1856,1971
5-24,373,1579,139,2091
50-99,50,278,5,333
>149,13,147,10,170


In [22]:
# 5.3.3 Targeted redundancy and numeric-distribution evidence
def cramers_v_5_3(table):
    table = table.loc[table.sum(axis=1).gt(0), table.sum(axis=0).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    observed = table.to_numpy(dtype=float)
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.sum()
    chi2 = ((observed - expected) ** 2 / expected).sum()
    return float(np.sqrt((chi2 / observed.sum()) / min(table.shape[0] - 1, table.shape[1] - 1)))

manufactured_data_5_3 = analysis_5_3.loc[analysis_5_3["is_manufactured_context"]].copy()
mfg_status_table_5_3 = pd.crosstab(
    manufactured_data_5_3["manufactured_home_secured_property_type_status"],
    manufactured_data_5_3["manufactured_home_land_property_interest_status"],
)
mfg_informative_data_5_3 = manufactured_data_5_3.loc[
    manufactured_data_5_3["manufactured_home_secured_property_type_status"].eq("Informative")
    & manufactured_data_5_3["manufactured_home_land_property_interest_status"].eq("Informative")
]
mfg_informative_table_5_3 = pd.crosstab(
    mfg_informative_data_5_3["manufactured_home_secured_property_type"],
    mfg_informative_data_5_3["manufactured_home_land_property_interest"],
)
redundancy_evidence_5_3 = pd.DataFrame([
    {
        "block": "manufactured_home_fields",
        "population": "manufactured_context_statuses",
        "n": len(manufactured_data_5_3),
        "cramers_v": cramers_v_5_3(mfg_status_table_5_3),
    },
    {
        "block": "manufactured_home_fields",
        "population": "manufactured_context_informative_only",
        "n": len(mfg_informative_data_5_3),
        "cramers_v": cramers_v_5_3(mfg_informative_table_5_3),
    },
])

multifamily_data_5_3 = analysis_5_3.loc[analysis_5_3["is_multifamily_context"]].copy()
informative_multifamily_5_3 = multifamily_data_5_3.loc[
    multifamily_data_5_3["multifamily_affordable_units_status"].eq("Informative")
].copy()
informative_multifamily_5_3["affordable_units"] = pd.to_numeric(
    informative_multifamily_5_3["multifamily_affordable_units"], errors="coerce"
)
multifamily_value_summary_5_3 = (
    informative_multifamily_5_3.groupby("total_units", dropna=False)["affordable_units"]
    .agg(n="size", zero_n=lambda values: int(values.eq(0).sum()), median="median", max="max")
    .reset_index()
)
multifamily_numeric_validation_5_3 = pd.Series({
    "all_informative_values_parsed": informative_multifamily_5_3["affordable_units"].notna().all(),
    "all_informative_values_nonnegative": informative_multifamily_5_3["affordable_units"].ge(0).all(),
    "all_informative_values_integer": informative_multifamily_5_3["affordable_units"].mod(1).eq(0).all(),
})
display(mfg_status_table_5_3)
display(mfg_informative_table_5_3)
display(redundancy_evidence_5_3)
display(multifamily_value_summary_5_3)
display(multifamily_numeric_validation_5_3)

manufactured_home_land_property_interest_status,Exempt,Informative,Not Applicable
manufactured_home_secured_property_type_status,,,
Exempt,527,0,0
Informative,2,6258,2
Not Applicable,0,0,26


manufactured_home_land_property_interest,1,2,3,4
manufactured_home_secured_property_type,,,,
1,2427,10,6,0
2,517,41,3117,140


,block,population,n,cramers_v
0,manufactured_home_fields,manufactured_context_statuses,6815,0.980861
1,manufactured_home_fields,manufactured_context_informative_only,6258,0.839286


,total_units,n,zero_n,median,max
0,100-149,88,73,0.0,100
1,25-49,342,276,0.0,100
2,5-24,1579,1413,0.0,100
3,50-99,278,221,0.0,100
4,>149,147,108,0.0,100


all_informative_values_parsed         True
all_informative_values_nonnegative    True
all_informative_values_integer        True
dtype: bool

In [23]:
# 5.3.4 LEI and protected-metadata guardrails — descriptive only
def lei_diagnostic_5_3(feature, scope):
    status_column = f"{feature}_status"
    valid_lei_scope = scope.loc[scope["lei"].notna() & scope["lei"].astype(str).str.strip().ne("")].copy()
    diagnostic_value = valid_lei_scope[feature].where(
        valid_lei_scope[status_column].eq("Informative"), valid_lei_scope[status_column]
    )
    global_dominance = diagnostic_value.value_counts(dropna=False, normalize=True).iloc[0] * 100
    per_lei = pd.DataFrame({"lei": valid_lei_scope["lei"], "value": diagnostic_value}).groupby("lei")["value"].agg(
        row_n="size", dominant_share=lambda values: values.value_counts(normalize=True, dropna=False).iloc[0] * 100
    )
    informative_by_lei = valid_lei_scope.loc[valid_lei_scope[status_column].eq("Informative")].groupby("lei").size()
    informative_n = int(informative_by_lei.sum())
    return {
        "feature": feature,
        "scope_n": len(scope),
        "lei_n": int(per_lei.shape[0]),
        "global_dominant_share_pct": round(global_dominance, 4),
        "row_weighted_dominant_share_pct": round(np.average(per_lei["dominant_share"], weights=per_lei["row_n"]), 4),
        "informative_lei_n": int(informative_by_lei.shape[0]),
        "informative_top_1_lei_share_pct": 0.0 if informative_n == 0 else round(informative_by_lei.nlargest(1).sum() / informative_n * 100, 4),
        "informative_top_10_lei_share_pct": 0.0 if informative_n == 0 else round(informative_by_lei.nlargest(10).sum() / informative_n * 100, 4),
    }

lei_proxy_5_3 = pd.DataFrame([
    lei_diagnostic_5_3("manufactured_home_secured_property_type", manufactured_data_5_3),
    lei_diagnostic_5_3("manufactured_home_land_property_interest", manufactured_data_5_3),
    lei_diagnostic_5_3("multifamily_affordable_units", multifamily_data_5_3),
])
lei_proxy_5_3["lei_uplift_vs_global_pp"] = (
    lei_proxy_5_3["row_weighted_dominant_share_pct"] - lei_proxy_5_3["global_dominant_share_pct"]
).round(4)

def canonical_race_set_5_3(row):
    tokens = sorted({
        normalized_token_5_3(row[column]) for column in RACE_SLOT_COLUMNS_5_3
        if normalized_token_5_3(row[column]) not in SPECIAL_BLANK_5_3
    })
    return "|".join(tokens) if tokens else "blank_or_missing"

analysis_5_3["applicant_race_token_set_5_3"] = analysis_5_3.apply(canonical_race_set_5_3, axis=1)
manufactured_data_5_3["applicant_race_token_set_5_3"] = manufactured_data_5_3.apply(canonical_race_set_5_3, axis=1)
multifamily_data_5_3["applicant_race_token_set_5_3"] = multifamily_data_5_3.apply(canonical_race_set_5_3, axis=1)
protected_rows_5_3 = []
for feature, scope in [
    ("manufactured_home_secured_property_type", manufactured_data_5_3),
    ("manufactured_home_land_property_interest", manufactured_data_5_3),
    ("multifamily_affordable_units", multifamily_data_5_3),
]:
    status_column = f"{feature}_status"
    for protected_group in ["applicant_sex", "applicant_race_token_set_5_3"]:
        grouped = scope.groupby(protected_group, dropna=False)[status_column]
        for group_value, statuses in grouped:
            protected_rows_5_3.append({
                "feature": feature,
                "protected_group": protected_group,
                "group_value": group_value,
                "n": int(statuses.size),
                "informative_pct": round(statuses.eq("Informative").mean() * 100, 4),
                "special_or_not_applicable_pct": round(statuses.isin(["Exempt", "Not Applicable", "Blank"]).mean() * 100, 4),
            })
protected_sensitivity_5_3 = pd.DataFrame(protected_rows_5_3)
assert lei_proxy_5_3["lei_uplift_vs_global_pp"].notna().all()
assert not protected_sensitivity_5_3.empty
display(lei_proxy_5_3)
display(protected_sensitivity_5_3.sort_values(["feature", "protected_group", "n"], ascending=[True, True, False]))

,feature,scope_n,lei_n,global_dominant_share_pct,row_weighted_dominant_share_pct,informative_lei_n,informative_top_1_lei_share_pct,informative_top_10_lei_share_pct,lei_uplift_vs_global_pp
0,manufactured_home_secured_property_type,6815,186,55.9941,93.3382,149,21.6385,74.9760,37.3441
1,manufactured_home_land_property_interest,6815,186,45.8254,86.9699,148,21.6523,75.0240,41.1445
2,multifamily_affordable_units,3107,215,67.2996,88.9282,141,14.3796,48.8085,21.6286


,feature,protected_group,group_value,n,informative_pct,special_or_not_applicable_pct
85,manufactured_home_land_property_interest,applicant_race_token_set_5_3,5,5532,91.1605,8.8395
86,manufactured_home_land_property_interest,applicant_race_token_set_5_3,6,846,96.9267,3.0733
76,manufactured_home_land_property_interest,applicant_race_token_set_5_3,3,179,96.6480,3.3520
87,manufactured_home_land_property_interest,applicant_race_token_set_5_3,7,79,62.0253,37.9747
49,manufactured_home_land_property_interest,applicant_race_token_set_5_3,1,67,94.0299,5.9701
...,...,...,...,...,...,...
91,multifamily_affordable_units,applicant_sex,4,2559,77.6866,22.3134
88,multifamily_affordable_units,applicant_sex,1,313,76.3578,23.6422
90,multifamily_affordable_units,applicant_sex,3,184,92.9348,7.0652
89,multifamily_affordable_units,applicant_sex,2,50,70.0000,30.0000


In [24]:
# 5.3.5 Evidence-based screening record and timing-resolution closure gate
# FIG fields 89, 90, and 92 validate the reporting domain. No local source proves
# that these values are available at the approval-model prediction point, so the
# unresolved timing is closed by explicit exclusion rather than treated as passed.
timing_evidence_5_3 = pd.DataFrame([
    {
        "feature_or_block": feature,
        "timing_status": "requires_lineage_confirmation",
        "timing_evidence": (
            f"{SCHEMA_5_3[feature]['fig_reference']} validates report-domain values; "
            "no local evidence establishes prediction-time availability."
        ),
    }
    for feature in GROUP_5_3_COLUMNS
])

screening_decision_5_3 = pd.DataFrame({"feature_or_block": GROUP_5_3_COLUMNS}).merge(
    timing_evidence_5_3, on="feature_or_block", validate="one_to_one"
)
screening_decision_5_3["business_reason"] = screening_decision_5_3["feature_or_block"].map({
    "manufactured_home_secured_property_type": "Property collateral context for manufactured-home applications.",
    "manufactured_home_land_property_interest": "Land-interest context for manufactured-home applications.",
    "multifamily_affordable_units": "Affordable-unit count for applicable multifamily properties.",
})
screening_decision_5_3["quality_evidence"] = screening_decision_5_3["feature_or_block"].map(
    quality_5_3.set_index("feature")["informative_n"].map(
        lambda n: f"{int(n)} informative rows; schema audit reconciled."
    )
)
screening_decision_5_3["redundancy_evidence"] = screening_decision_5_3["feature_or_block"].map({
    "manufactured_home_secured_property_type": "Reviewed jointly with manufactured_home_land_property_interest in manufactured context.",
    "manufactured_home_land_property_interest": "Reviewed jointly with manufactured_home_secured_property_type in manufactured context.",
    "multifamily_affordable_units": "Distribution reviewed only within multifamily context; raw-vs-derived dwelling comparison remains separate.",
})
screening_decision_5_3["protected_group_sensitivity"] = (
    "Coverage/status diagnostics recorded by applicant_sex and canonical five-slot race token set."
)
screening_decision_5_3["screening_status"] = "exclude_from_further_review"
screening_decision_5_3["decision"] = "exclude"
screening_decision_5_3["decision_reason"] = (
    "Excluded because local prediction-time lineage is not established; reporting-domain validation alone does not prove availability before decision."
)
screening_decision_5_3["gsv_player"] = "not_applicable"
screening_decision_5_3["validation_evidence"] = (
    "not_run_not_required_for_data_governance_scope"
)
screening_decision_5_3["stability_evidence"] = "not_applicable"

timing_resolution_mask_5_3 = (
    screening_decision_5_3["timing_status"].eq(
        "available_before_decision_by_definition"
    )
    | (
        screening_decision_5_3["timing_status"].eq(
            "requires_lineage_confirmation"
        )
        & screening_decision_5_3["screening_status"].eq(
            "exclude_from_further_review"
        )
        & screening_decision_5_3["decision"].eq("exclude")
        & screening_decision_5_3["decision_reason"].notna()
    )
)
decision_record_mask_5_3 = (
    screening_decision_5_3["screening_status"].eq(
        "exclude_from_further_review"
    )
    & screening_decision_5_3["decision"].eq("exclude")
    & screening_decision_5_3["gsv_player"].eq("not_applicable")
    & screening_decision_5_3["validation_evidence"].eq(
        "not_run_not_required_for_data_governance_scope"
    )
    & screening_decision_5_3["stability_evidence"].eq("not_applicable")
)

section_5_3_gate = pd.Series({
    "prior_5_2_gate": bool(section_5_2_gate["ready_to_start_5_3"]),
    "schema_gate": bool(quality_validation_5_3[["all_counts_reconciled", "no_invalid_tokens", "no_blank_tokens"]].all()),
    "numeric_gate": bool(multifamily_numeric_validation_5_3.all()),
    "applicability_gate": bool(applicability_validation_5_3["no_outside_context_informative_values"]),
    "applicability_exceptions_recorded": bool(applicability_validation_5_3["context_exceptions_recorded"]),
    "redundancy_evidence_gate": redundancy_evidence_5_3["cramers_v"].notna().all(),
    "lei_diagnostic_gate": lei_proxy_5_3["lei_uplift_vs_global_pp"].notna().all(),
    "protected_sensitivity_gate": not protected_sensitivity_5_3.empty,
    "timing_resolution_gate": bool(timing_resolution_mask_5_3.all()),
    "decision_record_gate": bool(decision_record_mask_5_3.all()),
})
section_5_3_gate["ready_to_close_5_3"] = bool(section_5_3_gate.all())
section_5_3_gate["ready_to_start_5_4"] = bool(
    section_5_3_gate["ready_to_close_5_3"]
)

assert section_5_3_gate[[
    "schema_gate", "numeric_gate", "applicability_gate",
    "lei_diagnostic_gate", "protected_sensitivity_gate",
]].all()
assert timing_resolution_mask_5_3.all()
assert decision_record_mask_5_3.all()
assert screening_decision_5_3["decision_reason"].notna().all()
assert section_5_3_gate["ready_to_close_5_3"]
assert section_5_3_gate["ready_to_start_5_4"]
display(section_5_3_gate)
display(screening_decision_5_3)


prior_5_2_gate                       True
schema_gate                          True
numeric_gate                         True
applicability_gate                   True
applicability_exceptions_recorded    True
redundancy_evidence_gate             True
lei_diagnostic_gate                  True
protected_sensitivity_gate           True
timing_resolution_gate               True
decision_record_gate                 True
ready_to_close_5_3                   True
ready_to_start_5_4                   True
dtype: bool

,feature_or_block,timing_status,timing_evidence,business_reason,quality_evidence,redundancy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence
0,manufactured_home_secured_property_type,requires_lineage_confirmation,2024 HMDA FIG field 89 validates report-domain...,Property collateral context for manufactured-h...,6262 informative rows; schema audit reconciled.,Reviewed jointly with manufactured_home_land_p...,Coverage/status diagnostics recorded by applic...,exclude_from_further_review,exclude,Excluded because local prediction-time lineage...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
1,manufactured_home_land_property_interest,requires_lineage_confirmation,2024 HMDA FIG field 90 validates report-domain...,Land-interest context for manufactured-home ap...,6258 informative rows; schema audit reconciled.,Reviewed jointly with manufactured_home_secure...,Coverage/status diagnostics recorded by applic...,exclude_from_further_review,exclude,Excluded because local prediction-time lineage...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
2,multifamily_affordable_units,requires_lineage_confirmation,2024 HMDA FIG field 92 validates report-domain...,Affordable-unit count for applicable multifami...,2434 informative rows; schema audit reconciled.,Distribution reviewed only within multifamily ...,Coverage/status diagnostics recorded by applic...,exclude_from_further_review,exclude,Excluded because local prediction-time lineage...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable


### 5.3.1 Kết luận screening

**Observed facts** — Cả ba fields reconcile đủ 293.301 dòng, không có blank hoặc invalid token. Informative counts lần lượt là 6.262 (`manufactured_home_secured_property_type`), 6.258 (`manufactured_home_land_property_interest`) và 2.434 (`multifamily_affordable_units`). Không có informative value ngoài applicable context. Các special/context exceptions được giữ riêng: 26, 28 và 163 dòng Not Applicable trong context tương ứng; 527, 529 và 510 dòng Exempt trong context tương ứng.

**Redundancy and guardrails** — Hai manufactured-home fields có Cramér's V = 0,9809 khi tính status trong manufactured context và 0,8393 khi chỉ tính informative values; vì vậy chúng chỉ được xem xét như một joint business block. Với 2.434 multifamily informative rows, mọi giá trị đều parse được, không âm và là integer; 2.091 dòng bằng 0. LEI row-weighted dominance vượt global dominance lần lượt 37,3441; 41,1445; và 21,6286 điểm phần trăm. Đây là diagnostic proxy-risk, không phải quy tắc tự động loại feature. Protected diagnostics dùng `applicant_sex` và canonical set của cả năm race slots, chỉ mô tả coverage/status.

**Timing resolution and decision** — FIG fields 89, 90 và 92 chỉ xác minh reporting domain; không tìm thấy local lineage chứng minh ba field có sẵn trước prediction decision. Timing được resolve bằng quyết định rõ ràng, không phải đổi một gate thất bại thành bằng chứng timing đạt. Cả ba field được đặt `screening_status=exclude_from_further_review`, `decision=exclude`, `gsv_player=not_applicable`, `validation_evidence=not_run_not_required_for_data_governance_scope`, và `stability_evidence=not_applicable`.

**Closure outcome** — `timing_resolution_gate=True` chỉ khi từng dòng có timing đã xác nhận hoặc có quyết định exclude vì timing chưa xác nhận. Tất cả decision records đã hoàn chỉnh, nên `ready_to_close_5_3=True` và `ready_to_start_5_4=True`. Không có kết luận về model performance, feature importance, fairness outcome hoặc quan hệ nhân quả.


### 5.4. Credit-score model types — joint-block screening

**Phạm vi:** `applicant_credit_score_type` và `co-applicant_credit_score_type` trên toàn bộ cohort 293.301 dòng.

Đây là categorical model-name/version codes, không phải numeric credit score. Code `9` là Not Applicable; `10` chỉ hợp lệ cho co-applicant và nghĩa là không có co-applicant; `1111` là Exempt. Code `8` phụ thuộc conditional free-form model name theo FIG, nhưng companion free-form field không tồn tại trong public schema 99 cột và không được đưa vào model. `lei`, `applicant_sex` và năm race slots chỉ là diagnostic metadata.

In [25]:
# 5.4.1 Raw-token inventory, schema, and special-state audit
GROUP_5_4_COLUMNS=["applicant_credit_score_type","co-applicant_credit_score_type"]
RACE_SLOT_COLUMNS_5_4=[f"applicant_race-{i}" for i in range(1,6)]
MODEL_CODES_5_4={"1":"Equifax Beacon 5.0","2":"Experian/Fair Isaac Risk Model v2","3":"FICO Risk Score Classic 04","4":"FICO Risk Score Classic 98","5":"VantageScore 2.0","6":"VantageScore 3.0","7":"Multiple models","8":"Other model; conditional free-form dependency","11":"FICO Score 9","12":"FICO Score 8","13":"FICO Score 10","14":"FICO Score 10T","15":"VantageScore 4.0"}
SPECIAL_CODES_5_4={"9":"Not Applicable","10":"No co-applicant","1111":"Exempt"}
def status_5_4(v,f):
 t=str(v).strip()
 if not t:return "blank"
 if t in MODEL_CODES_5_4:return "informative"
 if t=="9":return "not_applicable"
 if t=="1111":return "exempt"
 if t=="10" and f==GROUP_5_4_COLUMNS[1]:return "no_co_applicant"
 return "invalid"
required_5_4=GROUP_5_4_COLUMNS+["lei","applicant_sex"]+RACE_SLOT_COLUMNS_5_4
assert set(required_5_4).issubset(dataset_b.columns)
group_5_4_data=dataset_b[required_5_4].copy()
for f in GROUP_5_4_COLUMNS:group_5_4_data[f+"_status"]=group_5_4_data[f].map(lambda v,f=f:status_5_4(v,f))
raw_token_inventory_5_4=pd.concat([group_5_4_data[f].value_counts(dropna=False).rename_axis("raw_code").reset_index(name="n").assign(feature=f) for f in GROUP_5_4_COLUMNS],ignore_index=True)
raw_token_inventory_5_4["status"]=raw_token_inventory_5_4.apply(lambda r:status_5_4(r.raw_code,r.feature),axis=1)
raw_token_inventory_5_4["description"]=raw_token_inventory_5_4.raw_code.map({**MODEL_CODES_5_4,**SPECIAL_CODES_5_4}).fillna("Unrecognized or blank")
rows=[]
for f in GROUP_5_4_COLUMNS:
 s=group_5_4_data[f+"_status"]
 rows.append({"feature":f,"row_count":len(s),"informative_n":int(s.eq("informative").sum()),"not_applicable_n":int(s.eq("not_applicable").sum()),"no_co_applicant_n":int(s.eq("no_co_applicant").sum()),"exempt_n":int(s.eq("exempt").sum()),"blank_n":int(s.eq("blank").sum()),"invalid_n":int(s.eq("invalid").sum()),"code_8_n":int(group_5_4_data[f].eq("8").sum())})
quality_5_4=pd.DataFrame(rows); quality_5_4["informative_pct"]=(quality_5_4.informative_n/quality_5_4.row_count*100).round(4)
quality_5_4["reconciled_n"]=quality_5_4[["informative_n","not_applicable_n","no_co_applicant_n","exempt_n","blank_n","invalid_n"]].sum(axis=1)
free_form_dependency_5_4=pd.Series({"code_8_requires_conditional_free_form":True,"public_99_column_schema_exposes_companion_field":False,"free_form_field_admitted_to_model":False})
schema_validation_5_4=pd.Series({"full_cohort":len(group_5_4_data)==293301,"all_counts_reconciled":quality_5_4.reconciled_n.eq(len(group_5_4_data)).all(),"no_invalid_tokens":quality_5_4.invalid_n.eq(0).all(),"no_blank_tokens":quality_5_4.blank_n.eq(0).all(),"code_10_only_in_coapplicant":not group_5_4_data[GROUP_5_4_COLUMNS[0]].eq("10").any(),"code_8_dependency_recorded":bool(free_form_dependency_5_4.iloc[0] and not free_form_dependency_5_4.iloc[1] and not free_form_dependency_5_4.iloc[2])})
assert schema_validation_5_4.all()
display(raw_token_inventory_5_4.sort_values(["feature","raw_code"]));display(quality_5_4);display(free_form_dependency_5_4);display(schema_validation_5_4)

,raw_code,n,feature,status,description
1,1,58757,applicant_credit_score_type,informative,Equifax Beacon 5.0
7,11,8271,applicant_credit_score_type,informative,FICO Score 9
8,1111,8245,applicant_credit_score_type,exempt,Exempt
6,12,11682,applicant_credit_score_type,informative,FICO Score 8
11,14,640,applicant_credit_score_type,informative,FICO Score 10T
9,15,5815,applicant_credit_score_type,informative,VantageScore 4.0
0,2,59182,applicant_credit_score_type,informative,Experian/Fair Isaac Risk Model v2
2,3,53448,applicant_credit_score_type,informative,FICO Risk Score Classic 04
12,4,124,applicant_credit_score_type,informative,FICO Risk Score Classic 98
13,5,1,applicant_credit_score_type,informative,VantageScore 2.0


,feature,row_count,informative_n,not_applicable_n,no_co_applicant_n,exempt_n,blank_n,invalid_n,code_8_n,informative_pct,reconciled_n
0,applicant_credit_score_type,293301,247099,37957,0,8245,0,0,27359,84.2476,293301
1,co-applicant_credit_score_type,293301,55862,59364,169830,8245,0,0,4117,19.0460,293301


code_8_requires_conditional_free_form               True
public_99_column_schema_exposes_companion_field    False
free_form_field_admitted_to_model                  False
dtype: bool

full_cohort                    True
all_counts_reconciled          True
no_invalid_tokens              True
no_blank_tokens                True
code_10_only_in_coapplicant    True
code_8_dependency_recorded     True
dtype: bool

In [26]:
# 5.4.2 Joint applicability, association, LEI, and protected coverage
def cramers_v_5_4(a,b):
 t=pd.crosstab(a,b,dropna=False);o=t.to_numpy(float);e=np.outer(o.sum(1),o.sum(0))/o.sum();return float(np.sqrt((((o-e)**2/e).sum()/o.sum())/min(o.shape[0]-1,o.shape[1]-1)))
A,C=GROUP_5_4_COLUMNS; AS=A+"_status"; CS=C+"_status"
joint_status_5_4=pd.crosstab(group_5_4_data[AS],group_5_4_data[CS],dropna=False)
both_applicable_5_4=group_5_4_data[group_5_4_data[AS].eq("informative")&group_5_4_data[CS].eq("informative")]
joint_summary_5_4=pd.Series({"row_count":len(group_5_4_data),"no_co_applicant_n":int(group_5_4_data[C].eq("10").sum()),"no_co_applicant_pct":round(group_5_4_data[C].eq("10").mean()*100,4),"both_informative_n":len(both_applicable_5_4),"both_informative_exact_match_pct":round(both_applicable_5_4[A].eq(both_applicable_5_4[C]).mean()*100,4),"applicant_coapplicant_cramers_v_applicable":round(cramers_v_5_4(both_applicable_5_4[A],both_applicable_5_4[C]),6)})
def lei_diag_5_4(f):
 st=f+"_status";x=group_5_4_data[group_5_4_data.lei.str.strip().ne("")].copy();v=x[f].where(x[st].eq("informative"),x[st]);glob=v.value_counts(normalize=True,dropna=False).iloc[0]*100
 z=pd.DataFrame({"lei":x.lei,"v":v}).groupby("lei").v.agg(n="size",dom=lambda s:s.value_counts(normalize=True,dropna=False).iloc[0]*100);w=np.average(z.dom,weights=z.n);info=x[x[st].eq("informative")].groupby("lei").size();tot=info.sum()
 return {"feature":f,"lei_n":len(z),"global_dominant_share_pct":round(glob,4),"row_weighted_dominant_share_pct":round(w,4),"lei_uplift_vs_global_pp":round(w-glob,4),"informative_lei_n":len(info),"informative_top_1_lei_share_pct":round(info.nlargest(1).sum()/tot*100,4),"informative_top_10_lei_share_pct":round(info.nlargest(10).sum()/tot*100,4)}
lei_proxy_5_4=pd.DataFrame([lei_diag_5_4(f) for f in GROUP_5_4_COLUMNS])
def race_set_5_4(r):
 x=sorted({str(r[c]).strip() for c in RACE_SLOT_COLUMNS_5_4 if str(r[c]).strip()});return "|".join(x) if x else "blank_or_missing"
group_5_4_data["race_set_5_4"]=group_5_4_data.apply(race_set_5_4,axis=1)
rows=[]
for f in GROUP_5_4_COLUMNS:
 for g in ["applicant_sex","race_set_5_4"]:
  for value,s in group_5_4_data.groupby(g,dropna=False)[f+"_status"]:
   rows.append({"feature":f,"protected_group":g,"group_value":value,"n":int(s.size),"informative_pct":round(s.eq("informative").mean()*100,4),"special_or_unavailable_pct":round((~s.eq("informative")).mean()*100,4)})
protected_sensitivity_5_4=pd.DataFrame(rows)
protected_summary_5_4=protected_sensitivity_5_4[protected_sensitivity_5_4.n.ge(100)].groupby(["feature","protected_group"]).agg(groups_n=("group_value","size"),informative_min_pct=("informative_pct","min"),informative_max_pct=("informative_pct","max")).reset_index()
assert len(both_applicable_5_4)>0 and pd.notna(joint_summary_5_4.iloc[-1]);assert lei_proxy_5_4.lei_uplift_vs_global_pp.notna().all();assert not protected_sensitivity_5_4.empty
display(joint_status_5_4);display(joint_summary_5_4);display(lei_proxy_5_4);display(protected_summary_5_4)

co-applicant_credit_score_type_status,exempt,informative,no_co_applicant,not_applicable
applicant_credit_score_type_status,,,,
exempt,8245,0,0,0
informative,0,37886,158259,50954
not_applicable,0,17976,11571,8410


row_count                                     293301.000000
no_co_applicant_n                             169830.000000
no_co_applicant_pct                               57.903000
both_informative_n                             37886.000000
both_informative_exact_match_pct                  70.669900
applicant_coapplicant_cramers_v_applicable         0.869449
dtype: float64

,feature,lei_n,global_dominant_share_pct,row_weighted_dominant_share_pct,lei_uplift_vs_global_pp,informative_lei_n,informative_top_1_lei_share_pct,informative_top_10_lei_share_pct
0,applicant_credit_score_type,715,20.1779,55.8979,35.7200,509,5.9413,37.7670
1,co-applicant_credit_score_type,715,57.9030,62.4747,4.5718,367,6.5698,34.7374


,feature,protected_group,groups_n,informative_min_pct,informative_max_pct
0,applicant_credit_score_type,applicant_sex,5,0.0992,87.3551
1,applicant_credit_score_type,race_set_5_4,27,0.3653,95.2663
2,co-applicant_credit_score_type,applicant_sex,5,0.9821,41.2060
3,co-applicant_credit_score_type,race_set_5_4,27,1.0366,26.0355


In [27]:
# 5.4.3 Timing, concept assessment, joint-block decision, and closure gate
timing_status_5_4="reporting_definition_supports_predecision_context_but_local_lineage_not_established"
timing_evidence_5_4="2024 HMDA FIG 4(a)(15) identifies the model used to generate the reported credit score; local prediction-time lineage is not established."
concept_assessment_5_4="Model name/version describes scoring and reporting process, not borrower credit-score magnitude. Applicant model type has substantial LEI concentration, and code 8 lacks its conditional companion in the public schema."
q=quality_5_4.set_index("feature");l=lei_proxy_5_4.set_index("feature")
screening_decision_5_4=pd.DataFrame({"feature_or_block":GROUP_5_4_COLUMNS})
screening_decision_5_4["business_reason"]="Candidate credit-score-model identity; reviewed only as one joint block."
screening_decision_5_4["timing_status"]=timing_status_5_4;screening_decision_5_4["timing_evidence"]=timing_evidence_5_4
screening_decision_5_4["quality_evidence"]=screening_decision_5_4.feature_or_block.map({f:f"{int(q.loc[f,'informative_n'])} informative rows ({q.loc[f,'informative_pct']:.4f}%); {int(q.loc[f,'code_8_n'])} code-8 rows; all states reconciled." for f in GROUP_5_4_COLUMNS})
screening_decision_5_4["applicability_evidence"]=screening_decision_5_4.feature_or_block.map({A:"Code 9 is Not Applicable; code 10 is invalid for applicant.",C:f"Code 10 records no co-applicant in {int(joint_summary_5_4.no_co_applicant_n)} rows ({joint_summary_5_4.no_co_applicant_pct:.4f}%)."})
screening_decision_5_4["redundancy_evidence"]=f"Joint applicable n={int(joint_summary_5_4.both_informative_n)}; Cramer's V={joint_summary_5_4.applicant_coapplicant_cramers_v_applicable:.6f}; exact match={joint_summary_5_4.both_informative_exact_match_pct:.4f}%."
screening_decision_5_4["organization_proxy_evidence"]=screening_decision_5_4.feature_or_block.map({f:f"LEI weighted dominance {l.loc[f,'row_weighted_dominant_share_pct']:.4f}%; global {l.loc[f,'global_dominant_share_pct']:.4f}%; uplift {l.loc[f,'lei_uplift_vs_global_pp']:+.4f} pp." for f in GROUP_5_4_COLUMNS})
screening_decision_5_4["protected_group_sensitivity"]="Descriptive sex and canonical race-set coverage completed; not a decision rule."
screening_decision_5_4["concept_assessment"]=concept_assessment_5_4
screening_decision_5_4["screening_status"]="exclude_from_further_review";screening_decision_5_4["decision"]="exclude"
screening_decision_5_4["decision_reason"]="Excluded jointly because these fields mainly encode scoring/reporting process, local timing lineage is not established, and code 8's conditional companion is absent from the public schema."
screening_decision_5_4["gsv_player"]="not_applicable";screening_decision_5_4["validation_evidence"]="not_run_not_required_for_data_governance_scope";screening_decision_5_4["stability_evidence"]="not_applicable"
timing_resolution_5_4=screening_decision_5_4.decision.eq("exclude")&screening_decision_5_4.decision_reason.notna()
fields_5_4=["business_reason","timing_status","timing_evidence","quality_evidence","applicability_evidence","redundancy_evidence","organization_proxy_evidence","protected_group_sensitivity","concept_assessment","screening_status","decision","decision_reason","gsv_player","validation_evidence","stability_evidence"]
section_5_4_gate=pd.Series({"prior_5_3_gate":bool(section_5_3_gate.ready_to_start_5_4),"schema_gate":bool(schema_validation_5_4.all()),"special_state_gate":quality_5_4[["not_applicable_n","exempt_n"]].notna().all().all(),"code_8_dependency_gate":bool(schema_validation_5_4.code_8_dependency_recorded),"joint_pattern_gate":int(joint_status_5_4.to_numpy().sum())==len(group_5_4_data),"applicable_association_gate":pd.notna(joint_summary_5_4.applicant_coapplicant_cramers_v_applicable),"timing_resolution_gate":bool(timing_resolution_5_4.all()),"lei_diagnostic_gate":lei_proxy_5_4.lei_uplift_vs_global_pp.notna().all(),"protected_coverage_gate":not protected_sensitivity_5_4.empty,"concept_assessment_gate":screening_decision_5_4.concept_assessment.notna().all(),"joint_block_decision_gate":screening_decision_5_4.decision.eq("exclude").all() and screening_decision_5_4.gsv_player.eq("not_applicable").all(),"decision_record_gate":screening_decision_5_4[fields_5_4].notna().all().all()})
section_5_4_gate["ready_to_close_5_4"]=bool(section_5_4_gate.all());section_5_4_gate["ready_to_start_5_5"]=bool(section_5_4_gate.ready_to_close_5_4)
assert section_5_4_gate.ready_to_close_5_4 and section_5_4_gate.ready_to_start_5_5
display(section_5_4_gate);display(screening_decision_5_4)

prior_5_3_gate                 True
schema_gate                    True
special_state_gate             True
code_8_dependency_gate         True
joint_pattern_gate             True
applicable_association_gate    True
timing_resolution_gate         True
lei_diagnostic_gate            True
protected_coverage_gate        True
concept_assessment_gate        True
joint_block_decision_gate      True
decision_record_gate           True
ready_to_close_5_4             True
ready_to_start_5_5             True
dtype: bool

,feature_or_block,business_reason,timing_status,timing_evidence,quality_evidence,applicability_evidence,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,concept_assessment,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence
0,applicant_credit_score_type,Candidate credit-score-model identity; reviewe...,reporting_definition_supports_predecision_cont...,2024 HMDA FIG 4(a)(15) identifies the model us...,247099 informative rows (84.2476%); 27359 code...,Code 9 is Not Applicable; code 10 is invalid f...,Joint applicable n=37886; Cramer's V=0.869449;...,LEI weighted dominance 55.8979%; global 20.177...,Descriptive sex and canonical race-set coverag...,Model name/version describes scoring and repor...,exclude_from_further_review,exclude,Excluded jointly because these fields mainly e...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
1,co-applicant_credit_score_type,Candidate credit-score-model identity; reviewe...,reporting_definition_supports_predecision_cont...,2024 HMDA FIG 4(a)(15) identifies the model us...,55862 informative rows (19.0460%); 4117 code-8...,Code 10 records no co-applicant in 169830 rows...,Joint applicable n=37886; Cramer's V=0.869449;...,LEI weighted dominance 62.4747%; global 57.903...,Descriptive sex and canonical race-set coverag...,Model name/version describes scoring and repor...,exclude_from_further_review,exclude,Excluded jointly because these fields mainly e...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable


### 5.4.4 Kết luận screening

**Schema và applicability** — Hai field reconcile đủ 293.301 dòng, không có blank hoặc invalid token. Applicant có 247.099 informative rows (84,2476%); co-applicant có 55.862 (19,0460%). Code `10` xuất hiện 169.830 lần (57,9030%) ở co-applicant và được giữ riêng như No co-applicant. `9` và `1111` được giữ như Not Applicable và Exempt. Các code `11`, `12`, `14`, `15` là schema-valid model codes, không phải numeric scores.

**Code 8 dependency** — Code `8` xuất hiện 27.359 lần ở applicant và 4.117 lần ở co-applicant. FIG yêu cầu conditional free-form model name, nhưng public schema 99 cột không expose companion field; field đó không được bổ sung vào model.

**Joint block và proxy diagnostics** — Trên 37.886 dòng cả hai field informative, Cramér's V là 0,869449 và exact raw-code match là 70,6699%. Applicant LEI weighted dominance cao hơn global 35,7200 điểm phần trăm; co-applicant uplift là 4,5718 điểm. Protected metadata chỉ mô tả coverage/status.

**Decision** — Hai field mô tả model name/version và scoring/reporting process, không phải magnitude của credit score. Do local prediction-time lineage chưa được thiết lập, code `8` thiếu companion dependency trong public schema và process-proxy risk đáng kể, joint block bị `exclude_from_further_review`. Cả hai dòng có `decision=exclude`, `gsv_player=not_applicable`, `validation_evidence=not_run_not_required_for_data_governance_scope`, `stability_evidence=not_applicable`.

**Closure outcome** — Mọi evidence/decision field đầy đủ; `ready_to_close_5_4=True` và `ready_to_start_5_5=True`. Không dùng model performance, feature importance, split hoặc `record_id`.

### 5.5. AUS slots — multi-valued block screening

**Phạm vi:** `aus-1` đến `aus-5` trên toàn bộ cohort 293.301 dòng. Năm slot là một multi-valued Automated Underwriting System concept; slot position không có ordinal business meaning.

`aus-1` cho phép system codes `1–5`, `7`, `6=Not applicable`, và `1111=Exempt`. Các slot 2–5 cho phép system codes `1–5`, `7` hoặc blank. Code `5` được giữ như Other system. Canonical unordered set chỉ phục vụ audit/redundancy và không trở thành derived model feature. Phần này chỉ kiểm tra AUS type, không đưa AUS result vào scope.

In [28]:
# 5.5.1 Raw-token inventory and slot-specific schema validation
AUS_COLUMNS_5_5=[f"aus-{i}" for i in range(1,6)]
RACE_SLOT_COLUMNS_5_5=[f"applicant_race-{i}" for i in range(1,6)]
AUS_SYSTEM_CODES_5_5={"1":"Desktop Underwriter","2":"Loan Prospector / Loan Product Advisor","3":"TOTAL Scorecard","4":"Guaranteed Underwriting System","5":"Other AUS","7":"Internal Proprietary System"}
def aus_status_5_5(value,slot):
 token=str(value).strip()
 if token=="":return "blank" if slot>1 else "invalid"
 if token in AUS_SYSTEM_CODES_5_5:return "other_system" if token=="5" else "informative"
 if slot==1 and token=="6":return "not_applicable"
 if slot==1 and token=="1111":return "exempt"
 return "invalid"
required_5_5=AUS_COLUMNS_5_5+["lei","applicant_sex"]+RACE_SLOT_COLUMNS_5_5
assert set(required_5_5).issubset(dataset_b.columns)
group_5_5_data=dataset_b[required_5_5].copy()
for slot,feature in enumerate(AUS_COLUMNS_5_5,1):group_5_5_data[feature+"_status"]=group_5_5_data[feature].map(lambda value,slot=slot:aus_status_5_5(value,slot))
raw_token_inventory_5_5=pd.concat([group_5_5_data[f].value_counts(dropna=False).rename_axis("raw_code").reset_index(name="n").assign(feature=f,slot=i) for i,f in enumerate(AUS_COLUMNS_5_5,1)],ignore_index=True)
raw_token_inventory_5_5["status"]=raw_token_inventory_5_5.apply(lambda row:aus_status_5_5(row.raw_code,row.slot),axis=1)
raw_token_inventory_5_5["description"]=raw_token_inventory_5_5.raw_code.map({**AUS_SYSTEM_CODES_5_5,"6":"Not applicable","1111":"Exempt","":"Blank later slot"}).fillna("Invalid")
quality_rows_5_5=[]
for slot,feature in enumerate(AUS_COLUMNS_5_5,1):
 statuses=group_5_5_data[feature+"_status"]
 quality_rows_5_5.append({"feature":feature,"slot":slot,"row_count":len(statuses),"informative_n":int(statuses.isin(["informative","other_system"]).sum()),"other_system_n":int(statuses.eq("other_system").sum()),"not_applicable_n":int(statuses.eq("not_applicable").sum()),"exempt_n":int(statuses.eq("exempt").sum()),"blank_n":int(statuses.eq("blank").sum()),"invalid_n":int(statuses.eq("invalid").sum())})
quality_5_5=pd.DataFrame(quality_rows_5_5);quality_5_5["reconciled_n"]=quality_5_5[["informative_n","not_applicable_n","exempt_n","blank_n","invalid_n"]].sum(axis=1)
schema_validation_5_5=pd.Series({"full_cohort":len(group_5_5_data)==293301,"all_counts_reconciled":quality_5_5.reconciled_n.eq(len(group_5_5_data)).all(),"no_invalid_tokens":quality_5_5.invalid_n.eq(0).all(),"aus_1_has_no_blank":quality_5_5.loc[quality_5_5.feature.eq("aus-1"),"blank_n"].eq(0).all(),"later_slots_use_blank_not_special_6_1111":not group_5_5_data[AUS_COLUMNS_5_5[1:]].isin(["6","1111"]).any().any()})
assert schema_validation_5_5.all()
display(raw_token_inventory_5_5.sort_values(["slot","raw_code"]));display(quality_5_5);display(schema_validation_5_5)

,raw_code,n,feature,slot,status,description
1,1,72446,aus-1,1,informative,Desktop Underwriter
5,1111,8670,aus-1,1,exempt,Exempt
2,2,36912,aus-1,1,informative,Loan Prospector / Loan Product Advisor
4,3,13329,aus-1,1,informative,TOTAL Scorecard
7,4,396,aus-1,1,informative,Guaranteed Underwriting System
3,5,15496,aus-1,1,other_system,Other AUS
0,6,138228,aus-1,1,not_applicable,Not applicable
6,7,7824,aus-1,1,informative,Internal Proprietary System
8,,265116,aus-2,2,blank,Blank later slot
10,1,7961,aus-2,2,informative,Desktop Underwriter


,feature,slot,row_count,informative_n,other_system_n,not_applicable_n,exempt_n,blank_n,invalid_n,reconciled_n
0,aus-1,1,293301,146403,15496,138228,8670,0,0,293301
1,aus-2,2,293301,28185,261,0,0,265116,0,293301
2,aus-3,3,293301,8949,6049,0,0,284352,0,293301
3,aus-4,4,293301,1902,1,0,0,291399,0,293301
4,aus-5,5,293301,1765,0,0,0,291536,0,293301


full_cohort                                 True
all_counts_reconciled                       True
no_invalid_tokens                           True
aus_1_has_no_blank                          True
later_slots_use_blank_not_special_6_1111    True
dtype: bool

In [29]:
# 5.5.2 Slot structure and canonical unordered audit set
filled_5_5=group_5_5_data[AUS_COLUMNS_5_5].apply(lambda column:column.str.strip().ne(""))
slot_gap_mask_5_5=pd.Series(False,index=group_5_5_data.index)
for position in range(1,len(AUS_COLUMNS_5_5)):slot_gap_mask_5_5|=filled_5_5.iloc[:,position]&~filled_5_5.iloc[:,:position].all(axis=1)
INFORMATIVE_AUS_CODES_5_5=set(AUS_SYSTEM_CODES_5_5)
def reported_aus_values_5_5(row):return [str(row[column]).strip() for column in AUS_COLUMNS_5_5 if str(row[column]).strip() in INFORMATIVE_AUS_CODES_5_5]
def canonical_aus_set_5_5(row):return tuple(sorted(set(reported_aus_values_5_5(row)),key=int))
reported_values_5_5=group_5_5_data.apply(reported_aus_values_5_5,axis=1);canonical_aus_set_audit_5_5=group_5_5_data.apply(canonical_aus_set_5_5,axis=1)
system_count_5_5=reported_values_5_5.map(len);duplicate_system_mask_5_5=reported_values_5_5.map(lambda values:len(values)!=len(set(values)));canonical_aus_set_text_5_5=canonical_aus_set_audit_5_5.map(lambda values:"|".join(values));any_system_n_5_5=int(system_count_5_5.gt(0).sum())
only_aus_1_mask_5_5=group_5_5_data["aus-1"].isin(INFORMATIVE_AUS_CODES_5_5)&~group_5_5_data[AUS_COLUMNS_5_5[1:]].isin(INFORMATIVE_AUS_CODES_5_5).any(axis=1)
slot_structure_5_5=pd.Series({"slot_gap_n":int(slot_gap_mask_5_5.sum()),"slot_gap_pct":round(slot_gap_mask_5_5.mean()*100,6),"duplicate_system_rows_n":int(duplicate_system_mask_5_5.sum()),"duplicate_system_rows_pct":round(duplicate_system_mask_5_5.mean()*100,6),"any_system_n":any_system_n_5_5,"any_system_pct":round(any_system_n_5_5/len(group_5_5_data)*100,4),"only_aus_1_n":int(only_aus_1_mask_5_5.sum()),"only_aus_1_pct_among_system_rows":round(only_aus_1_mask_5_5.sum()/any_system_n_5_5*100,4),"multi_system_n":int(system_count_5_5.gt(1).sum()),"canonical_set_is_audit_only":True,"canonical_set_admitted_as_model_feature":False})
system_count_distribution_5_5=system_count_5_5.value_counts().sort_index().rename_axis("reported_system_count").reset_index(name="n")
slot_filled_counts_5_5=filled_5_5.sum().rename_axis("feature").reset_index(name="filled_n")
canonical_set_frequency_5_5=canonical_aus_set_text_5_5.value_counts().rename_axis("canonical_unordered_set").reset_index(name="n")
slot_structure_resolution_5_5=pd.Series({"slot_order_evidence_recorded":slot_structure_5_5[["slot_gap_n","slot_gap_pct"]].notna().all(),"duplicate_evidence_recorded":slot_structure_5_5[["duplicate_system_rows_n","duplicate_system_rows_pct"]].notna().all(),"system_count_evidence_recorded":system_count_distribution_5_5.n.sum()==len(group_5_5_data),"canonical_set_audit_only":bool(slot_structure_5_5.canonical_set_is_audit_only and not slot_structure_5_5.canonical_set_admitted_as_model_feature)})
assert slot_structure_resolution_5_5.all()
display(slot_structure_5_5);display(system_count_distribution_5_5);display(slot_filled_counts_5_5);display(canonical_set_frequency_5_5.head(20));display(slot_structure_resolution_5_5)

slot_gap_n                                        2
slot_gap_pct                               0.000682
duplicate_system_rows_n                        2978
duplicate_system_rows_pct                  1.015339
any_system_n                                 146403
any_system_pct                              49.9156
only_aus_1_n                                 118216
only_aus_1_pct_among_system_rows             80.747
multi_system_n                                28187
canonical_set_is_audit_only                    True
canonical_set_admitted_as_model_feature       False
dtype: object

,reported_system_count,n
0,0,146898
1,1,118216
2,2,19239
3,3,7046
4,4,138
5,5,1764


,feature,filled_n
0,aus-1,293301
1,aus-2,28185
2,aus-3,8949
3,aus-4,1902
4,aus-5,1765


,canonical_unordered_set,n
0,,146898
1,1,54457
2,2,28759
3,1|2,16531
4,5,15487
5,3,13045
6,7,7821
7,1|2|5,6050
8,1|7,1454
9,2|7,938


slot_order_evidence_recorded      True
duplicate_evidence_recorded       True
system_count_evidence_recorded    True
canonical_set_audit_only          True
dtype: bool

In [30]:
# 5.5.3 Block coverage, LEI concentration, and protected-group coverage
block_coverage_5_5=pd.Series({"row_count":len(group_5_5_data),"any_reported_system_n":int(system_count_5_5.gt(0).sum()),"any_reported_system_pct":round(system_count_5_5.gt(0).mean()*100,4),"not_applicable_n":int(group_5_5_data["aus-1_status"].eq("not_applicable").sum()),"exempt_n":int(group_5_5_data["aus-1_status"].eq("exempt").sum()),"other_system_any_slot_n":int(group_5_5_data[[f+"_status" for f in AUS_COLUMNS_5_5]].eq("other_system").any(axis=1).sum())})
block_pattern_5_5=canonical_aus_set_text_5_5.where(canonical_aus_set_text_5_5.ne(""),group_5_5_data["aus-1_status"])
lei_scope_5_5=pd.DataFrame({"lei":group_5_5_data.lei,"pattern":block_pattern_5_5});lei_scope_5_5=lei_scope_5_5[lei_scope_5_5.lei.str.strip().ne("")]
global_dominance_5_5=lei_scope_5_5.pattern.value_counts(normalize=True,dropna=False).iloc[0]*100
per_lei_5_5=lei_scope_5_5.groupby("lei").pattern.agg(row_n="size",dominant_share=lambda values:values.value_counts(normalize=True,dropna=False).iloc[0]*100);weighted_dominance_5_5=np.average(per_lei_5_5.dominant_share,weights=per_lei_5_5.row_n)
lei_proxy_5_5=pd.Series({"lei_n":len(per_lei_5_5),"global_dominant_share_pct":round(global_dominance_5_5,4),"row_weighted_dominant_share_pct":round(weighted_dominance_5_5,4),"lei_uplift_vs_global_pp":round(weighted_dominance_5_5-global_dominance_5_5,4)})
def race_set_5_5(row):
 values=sorted({str(row[column]).strip() for column in RACE_SLOT_COLUMNS_5_5 if str(row[column]).strip()});return "|".join(values) if values else "blank_or_missing"
group_5_5_data["race_set_5_5"]=group_5_5_data.apply(race_set_5_5,axis=1);group_5_5_data["aus_any_system_5_5"]=system_count_5_5.gt(0)
protected_rows_5_5=[]
for protected_group in ["applicant_sex","race_set_5_5"]:
 for group_value,values in group_5_5_data.groupby(protected_group,dropna=False)["aus_any_system_5_5"]:protected_rows_5_5.append({"protected_group":protected_group,"group_value":group_value,"n":int(values.size),"any_system_pct":round(values.mean()*100,4)})
protected_sensitivity_5_5=pd.DataFrame(protected_rows_5_5);protected_summary_5_5=protected_sensitivity_5_5[protected_sensitivity_5_5.n.ge(100)].groupby("protected_group").agg(groups_n=("group_value","size"),any_system_min_pct=("any_system_pct","min"),any_system_max_pct=("any_system_pct","max")).reset_index()
assert block_coverage_5_5.row_count==293301;assert pd.notna(lei_proxy_5_5.lei_uplift_vs_global_pp);assert not protected_sensitivity_5_5.empty
display(block_coverage_5_5);display(lei_proxy_5_5);display(protected_summary_5_5)

row_count                  293301.0000
any_reported_system_n      146403.0000
any_reported_system_pct        49.9156
not_applicable_n           138228.0000
exempt_n                     8670.0000
other_system_any_slot_n     21805.0000
dtype: float64

lei_n                              715.0000
global_dominant_share_pct           47.1284
row_weighted_dominant_share_pct     72.2105
lei_uplift_vs_global_pp             25.0821
dtype: float64

,protected_group,groups_n,any_system_min_pct,any_system_max_pct
0,applicant_sex,5,0.0099,52.4052
1,race_set_5_5,27,0.2468,77.5148


In [31]:
# 5.5.4 Timing, process-proxy assessment, block decision, and final review gate
timing_status_5_5="available_before_decision_by_definition_aus_type_only";timing_evidence_5_5="2024 HMDA FIG defines the AUS used to evaluate the application; AUS result is explicitly outside this screening scope."
concept_assessment_5_5="AUS type identifies underwriting technology and institutional process, not a distinct applicant credit-information attribute. Canonical block patterns are strongly concentrated by LEI."
screening_decision_5_5=pd.DataFrame({"feature_or_block":AUS_COLUMNS_5_5});screening_decision_5_5["business_reason"]="Raw member of one multi-valued Automated Underwriting System block; slot order has no separate business meaning.";screening_decision_5_5["timing_status"]=timing_status_5_5;screening_decision_5_5["timing_evidence"]=timing_evidence_5_5
screening_decision_5_5["quality_evidence"]=screening_decision_5_5.feature_or_block.map(quality_5_5.set_index("feature").apply(lambda row:f"{int(row.informative_n)} system-code rows; {int(row.blank_n)} blanks; {int(row.invalid_n)} invalid; counts reconciled.",axis=1))
screening_decision_5_5["applicability_evidence"]=f"At least one system in {int(block_coverage_5_5.any_reported_system_n)} rows ({block_coverage_5_5.any_reported_system_pct:.4f}%); aus-1 preserves Not Applicable and Exempt.";screening_decision_5_5["redundancy_evidence"]=f"One unordered audit set only; {int(slot_structure_5_5.duplicate_system_rows_n)} duplicate-code rows, {int(slot_structure_5_5.slot_gap_n)} slot-gap rows, {int(slot_structure_5_5.multi_system_n)} multi-system rows.";screening_decision_5_5["organization_proxy_evidence"]=f"Canonical pattern LEI weighted dominance {lei_proxy_5_5.row_weighted_dominant_share_pct:.4f}%; global {lei_proxy_5_5.global_dominant_share_pct:.4f}%; uplift {lei_proxy_5_5.lei_uplift_vs_global_pp:+.4f} pp.";screening_decision_5_5["protected_group_sensitivity"]="Descriptive sex and canonical race-set coverage completed; not a fairness or decision rule.";screening_decision_5_5["concept_assessment"]=concept_assessment_5_5
screening_decision_5_5["screening_status"]="exclude_from_further_review";screening_decision_5_5["decision"]="exclude";screening_decision_5_5["decision_reason"]="Excluded as one block because AUS type primarily encodes underwriting technology, institution, and reporting process rather than applicant credit information; AUS result is not in scope.";screening_decision_5_5["gsv_player"]="not_applicable";screening_decision_5_5["validation_evidence"]="not_run_not_required_for_data_governance_scope";screening_decision_5_5["stability_evidence"]="not_applicable"
fields_5_5=["business_reason","timing_status","timing_evidence","quality_evidence","applicability_evidence","redundancy_evidence","organization_proxy_evidence","protected_group_sensitivity","concept_assessment","screening_status","decision","decision_reason","gsv_player","validation_evidence","stability_evidence"]
section_5_5_gate=pd.Series({"prior_5_4_gate":bool(section_5_4_gate.ready_to_start_5_5),"schema_gate":bool(schema_validation_5_5.all()),"special_state_gate":quality_5_5[["not_applicable_n","exempt_n","blank_n","other_system_n"]].notna().all().all(),"slot_structure_evidence_gate":bool(slot_structure_resolution_5_5.all()),"canonical_set_audit_only_gate":bool(slot_structure_5_5.canonical_set_is_audit_only and not slot_structure_5_5.canonical_set_admitted_as_model_feature),"coverage_applicability_gate":block_coverage_5_5.notna().all(),"timing_leakage_gate":screening_decision_5_5.timing_status.eq(timing_status_5_5).all(),"aus_result_out_of_scope_gate":screening_decision_5_5.timing_evidence.str.contains("outside").all(),"lei_diagnostic_gate":pd.notna(lei_proxy_5_5.lei_uplift_vs_global_pp),"protected_coverage_gate":not protected_sensitivity_5_5.empty,"concept_assessment_gate":screening_decision_5_5.concept_assessment.notna().all(),"block_decision_gate":screening_decision_5_5.decision.eq("exclude").all() and screening_decision_5_5.gsv_player.eq("not_applicable").all(),"decision_record_gate":screening_decision_5_5[fields_5_5].notna().all().all()})
section_5_5_gate["ready_to_close_5_5"]=bool(section_5_5_gate.all());section_5_5_gate["ready_to_close_review_screening"]=bool(section_5_5_gate.ready_to_close_5_5)
assert screening_decision_5_5.feature_or_block.tolist()==AUS_COLUMNS_5_5;assert section_5_5_gate.ready_to_close_review_screening
display(section_5_5_gate);display(screening_decision_5_5)

prior_5_4_gate                     True
schema_gate                        True
special_state_gate                 True
slot_structure_evidence_gate       True
canonical_set_audit_only_gate      True
coverage_applicability_gate        True
timing_leakage_gate                True
aus_result_out_of_scope_gate       True
lei_diagnostic_gate                True
protected_coverage_gate            True
concept_assessment_gate            True
block_decision_gate                True
decision_record_gate               True
ready_to_close_5_5                 True
ready_to_close_review_screening    True
dtype: bool

,feature_or_block,business_reason,timing_status,timing_evidence,quality_evidence,applicability_evidence,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,concept_assessment,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence
0,aus-1,Raw member of one multi-valued Automated Under...,available_before_decision_by_definition_aus_ty...,2024 HMDA FIG defines the AUS used to evaluate...,146403 system-code rows; 0 blanks; 0 invalid; ...,At least one system in 146403 rows (49.9156%);...,One unordered audit set only; 2978 duplicate-c...,Canonical pattern LEI weighted dominance 72.21...,Descriptive sex and canonical race-set coverag...,AUS type identifies underwriting technology an...,exclude_from_further_review,exclude,Excluded as one block because AUS type primari...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
1,aus-2,Raw member of one multi-valued Automated Under...,available_before_decision_by_definition_aus_ty...,2024 HMDA FIG defines the AUS used to evaluate...,28185 system-code rows; 265116 blanks; 0 inval...,At least one system in 146403 rows (49.9156%);...,One unordered audit set only; 2978 duplicate-c...,Canonical pattern LEI weighted dominance 72.21...,Descriptive sex and canonical race-set coverag...,AUS type identifies underwriting technology an...,exclude_from_further_review,exclude,Excluded as one block because AUS type primari...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
2,aus-3,Raw member of one multi-valued Automated Under...,available_before_decision_by_definition_aus_ty...,2024 HMDA FIG defines the AUS used to evaluate...,8949 system-code rows; 284352 blanks; 0 invali...,At least one system in 146403 rows (49.9156%);...,One unordered audit set only; 2978 duplicate-c...,Canonical pattern LEI weighted dominance 72.21...,Descriptive sex and canonical race-set coverag...,AUS type identifies underwriting technology an...,exclude_from_further_review,exclude,Excluded as one block because AUS type primari...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
3,aus-4,Raw member of one multi-valued Automated Under...,available_before_decision_by_definition_aus_ty...,2024 HMDA FIG defines the AUS used to evaluate...,1902 system-code rows; 291399 blanks; 0 invali...,At least one system in 146403 rows (49.9156%);...,One unordered audit set only; 2978 duplicate-c...,Canonical pattern LEI weighted dominance 72.21...,Descriptive sex and canonical race-set coverag...,AUS type identifies underwriting technology an...,exclude_from_further_review,exclude,Excluded as one block because AUS type primari...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable
4,aus-5,Raw member of one multi-valued Automated Under...,available_before_decision_by_definition_aus_ty...,2024 HMDA FIG defines the AUS used to evaluate...,1765 system-code rows; 291536 blanks; 0 invali...,At least one system in 146403 rows (49.9156%);...,One unordered audit set only; 2978 duplicate-c...,Canonical pattern LEI weighted dominance 72.21...,Descriptive sex and canonical race-set coverag...,AUS type identifies underwriting technology an...,exclude_from_further_review,exclude,Excluded as one block because AUS type primari...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable


### 5.5.5 Kết luận screening

**Schema và slot structure** — Cả năm slot reconcile đủ 293.301 dòng và không có invalid token. `aus-1` luôn có giá trị; các slot sau dùng blank đúng schema. Có 2 hồ sơ có later-slot value sau một blank, 2.978 hồ sơ lặp system code và 28.187 hồ sơ báo cáo nhiều system. Các hiện tượng này được giữ làm audit evidence, không tự động sửa hoặc encode.

**Coverage** — Có 146.403 hồ sơ (49,9156%) báo cáo ít nhất một AUS system; trong số đó 118.216 hồ sơ (80,7470%) chỉ có một system. Canonical unordered set chỉ dùng cho audit/redundancy và không trở thành model feature.

**Proxy diagnostics** — Canonical block-pattern LEI weighted dominance là 72,2105%, cao hơn global dominance 25,0821 điểm phần trăm. Protected metadata chỉ mô tả coverage.

**Decision** — AUS type có trước decision theo reporting definition và AUS result không thuộc scope. Tuy nhiên block chủ yếu mô tả underwriting technology, institution và reporting process thay vì applicant credit information. Toàn bộ `aus-1`–`aus-5` bị `exclude_from_further_review`, với `decision=exclude`, `gsv_player=not_applicable`, `validation_evidence=not_run_not_required_for_data_governance_scope`, `stability_evidence=not_applicable`.

**Closure outcome** — Mọi evidence/decision field đầy đủ; `ready_to_close_5_5=True` và `ready_to_close_review_screening=True`. Không dùng model performance, feature importance, split hoặc `record_id`.

## 6. Data-governance closure of advanced 5.1–5.2 candidates

Phần này quay lại năm decision records đã mang nhãn `advance*`. Không chạy lại audit và không sử dụng model validation. Quyết định chỉ dùng saved full-cohort evidence về business meaning, timing, coverage/applicability, informative-population redundancy và LEI/process concentration.

Ba decision units là: `preapproval`; joint application-channel/institutional-relationship block; và joint interest-only/balloon repayment block.

In [32]:
# 6.1 Explicit data-only closure decisions for the five advanced records
ADVANCED_5_1_COLUMNS=["preapproval","submission_of_application","initially_payable_to_institution"]
ADVANCED_5_2_COLUMNS=["interest_only_payment","balloon_payment"]
for frame in [screening_decision_5_1,screening_decision_5_2]:
 for column in ["business_reason","validation_evidence","stability_evidence"]:
  if column not in frame.columns:frame[column]=pd.NA

preapproval_mask_4=screening_decision_5_1.feature_or_block.eq("preapproval")
application_process_mask_4=screening_decision_5_1.feature_or_block.isin(["submission_of_application","initially_payable_to_institution"])
repayment_mask_4=screening_decision_5_2.feature_or_block.isin(ADVANCED_5_2_COLUMNS)

screening_decision_5_1.loc[preapproval_mask_4,"business_reason"]="Distinct conditional application concept: whether a home-purchase application requested preapproval."
screening_decision_5_1.loc[preapproval_mask_4,"protected_group_sensitivity"]="Not used as a decision rule; Phase 4 closure relies on business, timing, applicability, redundancy, and LEI evidence."
screening_decision_5_1.loc[preapproval_mask_4,"screening_status"]="keep_after_data_governance_review"
screening_decision_5_1.loc[preapproval_mask_4,"decision"]="keep"
screening_decision_5_1.loc[preapproval_mask_4,"decision_reason"]="Kept because it is a distinct pre-decision application concept, has valid conditional applicability, is not redundant with loan_purpose, and has only +0.5844 pp LEI dominance uplift."
screening_decision_5_1.loc[preapproval_mask_4,"gsv_player"]="preapproval"

screening_decision_5_1.loc[application_process_mask_4,"business_reason"]="Joint application-channel and initial institutional-relationship block."
screening_decision_5_1.loc[application_process_mask_4,"protected_group_sensitivity"]="Not used as a decision rule; block is excluded for process/organization-proxy meaning."
screening_decision_5_1.loc[application_process_mask_4,"screening_status"]="exclude_from_further_review"
screening_decision_5_1.loc[application_process_mask_4,"decision"]="exclude"
screening_decision_5_1.loc[application_process_mask_4,"decision_reason"]="Excluded jointly because the fields primarily describe submission and institutional process rather than applicant or loan substance; informative-only V is 0.1735 and submission has +12.60 pp LEI dominance uplift."
screening_decision_5_1.loc[application_process_mask_4,"gsv_player"]="not_applicable"

screening_decision_5_2.loc[repayment_mask_4,"business_reason"]="Joint contractual special-repayment structure: interest-only and balloon-payment terms."
screening_decision_5_2.loc[repayment_mask_4,"screening_status"]="keep_after_data_governance_review"
screening_decision_5_2.loc[repayment_mask_4,"decision"]="keep"
screening_decision_5_2.loc[repayment_mask_4,"decision_reason"]="Kept jointly because both are pre-decision contractual terms with 97.05% informative coverage; informative-only association is 0.1701, so neither is redundant, and LEI concentration does not override their distinct loan-term meaning."
screening_decision_5_2.loc[repayment_mask_4,"gsv_player"]="special_repayment_structure"

for frame,mask in [(screening_decision_5_1,preapproval_mask_4|application_process_mask_4),(screening_decision_5_2,repayment_mask_4)]:
 frame.loc[mask,"validation_evidence"]="not_run_not_required_for_data_governance_scope"
 frame.loc[mask,"stability_evidence"]="data_level_evidence_reviewed_no_model_resampling"

phase_4_decision_records=pd.concat([screening_decision_5_1.loc[screening_decision_5_1.feature_or_block.isin(ADVANCED_5_1_COLUMNS)],screening_decision_5_2.loc[screening_decision_5_2.feature_or_block.isin(ADVANCED_5_2_COLUMNS)]],ignore_index=True,sort=False)
required_phase_4_fields=["business_reason","timing_status","timing_evidence","quality_evidence","conditional_applicability","redundancy_evidence","organization_proxy_evidence","protected_group_sensitivity","screening_status","decision","decision_reason","gsv_player","validation_evidence","stability_evidence"]
section_phase_4_gate=pd.Series({"prior_review_screening_gate":bool(section_5_5_gate.ready_to_close_review_screening),"five_records_present":phase_4_decision_records.feature_or_block.nunique()==5,"distinct_business_meaning_gate":phase_4_decision_records.business_reason.notna().all(),"timing_gate":phase_4_decision_records.timing_status.eq("available_before_decision_by_definition").all(),"coverage_applicability_gate":phase_4_decision_records[["quality_evidence","conditional_applicability"]].notna().all().all(),"informative_redundancy_gate":phase_4_decision_records.redundancy_evidence.notna().all(),"process_lei_gate":phase_4_decision_records.organization_proxy_evidence.notna().all(),"explicit_decision_gate":phase_4_decision_records.decision.isin(["keep","exclude"]).all(),"gsv_player_gate":phase_4_decision_records.gsv_player.notna().all(),"model_validation_out_of_scope_gate":phase_4_decision_records.validation_evidence.eq("not_run_not_required_for_data_governance_scope").all(),"decision_record_gate":phase_4_decision_records[required_phase_4_fields].notna().all().all()})
section_phase_4_gate["ready_to_close_advanced_candidates"]=bool(section_phase_4_gate.all())
assert phase_4_decision_records.loc[phase_4_decision_records.feature_or_block.eq("preapproval"),"decision"].eq("keep").all()
assert phase_4_decision_records.loc[phase_4_decision_records.feature_or_block.isin(["submission_of_application","initially_payable_to_institution"]),"decision"].eq("exclude").all()
assert phase_4_decision_records.loc[phase_4_decision_records.feature_or_block.isin(ADVANCED_5_2_COLUMNS),"decision"].eq("keep").all()
assert section_phase_4_gate.ready_to_close_advanced_candidates
display(phase_4_decision_records[required_phase_4_fields]);display(section_phase_4_gate)

,business_reason,timing_status,timing_evidence,quality_evidence,conditional_applicability,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence
0,Distinct conditional application concept: whet...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(4): whether the application...,100.00% valid; 100.00% informative; 0 invalid ...,All 9516 requested-preapproval rows are home p...,Conditional on home-purchase scope; not a dupl...,LEI row-weighted dominance 97.34%; global domi...,Not used as a decision rule; Phase 4 closure r...,keep_after_data_governance_review,keep,Kept because it is a distinct pre-decision app...,preapproval,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
1,Joint application-channel and initial institut...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(33): whether the applicant ...,100.00% valid; 97.18% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with initially_payable: full V=0.7...,LEI row-weighted dominance 96.30%; global domi...,Not used as a decision rule; block is excluded...,exclude_from_further_review,exclude,Excluded jointly because the fields primarily ...,not_applicable,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
2,Joint application-channel and initial institut...,available_before_decision_by_definition,"2024 HMDA FIG 4(a)(33): for an application, wh...",100.00% valid; 95.29% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with submission: full V=0.7178; in...,LEI row-weighted dominance 96.57%; global domi...,Not used as a decision rule; block is excluded...,exclude_from_further_review,exclude,Excluded jointly because the fields primarily ...,not_applicable,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
3,Joint contractual special-repayment structure:...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 19.29% code...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7173; informative-o...,LEI row-weighted dominance 89.37%; global 77.7...,Descriptive coverage only; not a decision rule...,keep_after_data_governance_review,keep,Kept jointly because both are pre-decision con...,special_repayment_structure,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling
4,Joint contractual special-repayment structure:...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 2.37% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7173; informative-o...,LEI row-weighted dominance 98.34%; global 94.6...,Descriptive coverage only; not a decision rule...,keep_after_data_governance_review,keep,Kept jointly because both are pre-decision con...,special_repayment_structure,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling


prior_review_screening_gate           True
five_records_present                  True
distinct_business_meaning_gate        True
timing_gate                           True
coverage_applicability_gate           True
informative_redundancy_gate           True
process_lei_gate                      True
explicit_decision_gate                True
gsv_player_gate                       True
model_validation_out_of_scope_gate    True
decision_record_gate                  True
ready_to_close_advanced_candidates    True
dtype: bool

### 6.2 Kết luận Phase 4

- `preapproval`: **keep**, `gsv_player=preapproval`.
- `submission_of_application` + `initially_payable_to_institution`: **exclude** như một process/institution block, `gsv_player=not_applicable`.
- `interest_only_payment` + `balloon_payment`: **keep** như một contractual repayment block, cùng `gsv_player=special_repayment_structure`.

Cả năm records ghi `validation_evidence=not_run_not_required_for_data_governance_scope`; không còn missing value trong decision, reason, player hoặc validation evidence. `ready_to_close_advanced_candidates=True`.

## 7. Final reconciliation of the 18 reviewed raw features

This section consolidates the saved evidence and decisions from Sections 5.1–5.5. It does not rerun an audit, train a model, or promote LEI/protected metadata to predictors.

In [33]:
# 7.1 Final 18-field audit, decision, and block reconciliation
EXPECTED_REVIEW_FIELDS = (
    GROUP_5_1_COLUMNS + GROUP_5_2_COLUMNS + GROUP_5_3_COLUMNS
    + GROUP_5_4_COLUMNS + AUS_COLUMNS_5_5
)
BLOCK_ID_BY_FIELD = {
    "preapproval": "preapproval",
    "submission_of_application": "application_submission_process",
    "initially_payable_to_institution": "application_submission_process",
    "conforming_loan_limit": "conforming_loan_limit_status",
    "negative_amortization": "negative_amortization",
    "interest_only_payment": "special_repayment_structure",
    "balloon_payment": "special_repayment_structure",
    "other_nonamortizing_features": "other_nonamortizing_features",
    "manufactured_home_secured_property_type": "manufactured_home_context",
    "manufactured_home_land_property_interest": "manufactured_home_context",
    "multifamily_affordable_units": "multifamily_affordable_units",
    "applicant_credit_score_type": "credit_score_model_type",
    "co-applicant_credit_score_type": "credit_score_model_type",
    **{field: "automated_underwriting_system" for field in AUS_COLUMNS_5_5},
}

def normalize_review_audit(frame, section, status_column="status"):
    result = frame.copy().rename(columns={
        "feature": "raw_field",
        status_column: "audit_state",
    })
    if "raw_code" not in result.columns:
        result["raw_code"] = pd.NA
    if "description" not in result.columns:
        result["description"] = result["audit_state"]
    if "share_pct" not in result.columns:
        result["share_pct"] = (
            result["n"] / result.groupby("raw_field")["n"].transform("sum") * 100
        ).round(4)
    result["audit_source"] = section
    return result[[
        "raw_field", "raw_code", "description", "audit_state",
        "n", "share_pct", "audit_source",
    ]]

audit_5_3_long = quality_5_3.melt(
    id_vars=["feature", "row_count"],
    value_vars=[
        "informative_n", "not_applicable_n", "exempt_n",
        "blank_n", "invalid_n",
    ],
    var_name="audit_state",
    value_name="n",
)
audit_5_3_long["audit_state"] = audit_5_3_long["audit_state"].str.removesuffix("_n")
audit_5_3_long["raw_code"] = pd.NA
audit_5_3_long["description"] = audit_5_3_long["audit_state"]
audit_5_3_long["share_pct"] = (
    audit_5_3_long["n"] / audit_5_3_long["row_count"] * 100
).round(4)

review_feature_audit = pd.concat([
    normalize_review_audit(audit_5_1_reviewed, "5.1"),
    normalize_review_audit(audit_5_2, "5.2"),
    normalize_review_audit(audit_5_3_long, "5.3", "audit_state"),
    normalize_review_audit(raw_token_inventory_5_4, "5.4"),
    normalize_review_audit(raw_token_inventory_5_5, "5.5"),
], ignore_index=True, sort=False)
review_feature_audit["block_id"] = review_feature_audit["raw_field"].map(
    BLOCK_ID_BY_FIELD
)
review_feature_audit = review_feature_audit[[
    "raw_field", "block_id", "audit_source", "raw_code",
    "description", "audit_state", "n", "share_pct",
]].sort_values(
    ["audit_source", "raw_field", "audit_state", "raw_code"],
    na_position="last",
).reset_index(drop=True)

decision_frames = []
for section, source in [
    ("5.1", screening_decision_5_1),
    ("5.2", screening_decision_5_2),
    ("5.3", screening_decision_5_3),
    ("5.4", screening_decision_5_4),
    ("5.5", screening_decision_5_5),
]:
    frame = source.copy().rename(columns={"feature_or_block": "raw_field"})
    frame["review_section"] = section
    decision_frames.append(frame)
review_feature_decision_table = pd.concat(
    decision_frames, ignore_index=True, sort=False
)
review_feature_decision_table["block_id"] = (
    review_feature_decision_table["raw_field"].map(BLOCK_ID_BY_FIELD)
)
BUSINESS_MEANING_BY_FIELD = {
    "preapproval": "Preapproval request for an applicable home-purchase application.",
    "submission_of_application": "Application-submission process.",
    "initially_payable_to_institution": "Initial institutional relationship.",
    "conforming_loan_limit": "Reported conforming-loan-limit status.",
    "negative_amortization": "Contractual negative-amortization term.",
    "interest_only_payment": "Contractual interest-only payment term.",
    "balloon_payment": "Contractual balloon-payment term.",
    "other_nonamortizing_features": "Other contractual non-amortizing term.",
    "manufactured_home_secured_property_type": "Manufactured-home collateral context.",
    "manufactured_home_land_property_interest": "Manufactured-home land-interest context.",
    "multifamily_affordable_units": "Applicable multifamily affordable-unit count.",
    "applicant_credit_score_type": "Applicant credit-score model identity.",
    "co-applicant_credit_score_type": "Co-applicant credit-score model identity.",
    **{
        field: "Raw slot in the multi-valued automated-underwriting-system block."
        for field in AUS_COLUMNS_5_5
    },
}
review_feature_decision_table["business_reason"] = (
    review_feature_decision_table["business_reason"].fillna(
        review_feature_decision_table["raw_field"].map(BUSINESS_MEANING_BY_FIELD)
    )
)
conditional = review_feature_decision_table.get(
    "conditional_applicability",
    pd.Series(pd.NA, index=review_feature_decision_table.index),
)
review_feature_decision_table["applicability_evidence"] = (
    review_feature_decision_table.get(
        "applicability_evidence",
        pd.Series(pd.NA, index=review_feature_decision_table.index),
    ).fillna(conditional).fillna(
        "Applicability and special states audited in the source section."
    )
)
review_feature_decision_table["validation_evidence"] = (
    review_feature_decision_table["validation_evidence"].fillna(
        "not_run_not_required_for_data_governance_scope"
    )
)
review_feature_decision_table["stability_evidence"] = (
    review_feature_decision_table["stability_evidence"].fillna("not_applicable")
)
review_feature_decision_table["lei_metadata_role"] = (
    "diagnostic_only_not_predictor"
)
review_feature_decision_table["protected_metadata_role"] = (
    "descriptive_coverage_only_not_predictor"
)
review_feature_decision_table = review_feature_decision_table[[
    "raw_field", "block_id", "review_section", "business_reason",
    "timing_status", "timing_evidence", "quality_evidence",
    "applicability_evidence", "redundancy_evidence",
    "organization_proxy_evidence", "protected_group_sensitivity",
    "screening_status", "decision", "decision_reason", "gsv_player",
    "validation_evidence", "stability_evidence",
    "lei_metadata_role", "protected_metadata_role",
]]

review_feature_reconciliation = (
    review_feature_decision_table.groupby("block_id", sort=False)
    .agg(
        field_count=("raw_field", "size"),
        raw_fields=("raw_field", lambda values: tuple(values)),
        decisions=("decision", lambda values: tuple(sorted(set(values)))),
        kept_field_count=("decision", lambda values: int(values.eq("keep").sum())),
        excluded_field_count=(
            "decision", lambda values: int(values.eq("exclude").sum())
        ),
        gsv_players=("gsv_player", lambda values: tuple(sorted(set(values)))),
    )
    .reset_index()
)
review_feature_reconciliation["joint_block"] = (
    review_feature_reconciliation["field_count"].gt(1)
)

observed_special_states = set(zip(
    review_feature_audit["raw_field"],
    review_feature_audit["audit_state"].str.lower(),
))
required_special_states = {
    *((field, "valid_special") for field in GROUP_5_2_COLUMNS),
    *((field, "not_applicable") for field in GROUP_5_3_COLUMNS),
    *((field, "exempt") for field in GROUP_5_3_COLUMNS),
    ("applicant_credit_score_type", "not_applicable"),
    ("applicant_credit_score_type", "exempt"),
    ("co-applicant_credit_score_type", "not_applicable"),
    ("co-applicant_credit_score_type", "no_co_applicant"),
    ("co-applicant_credit_score_type", "exempt"),
    ("aus-1", "not_applicable"),
    ("aus-1", "exempt"),
    ("aus-2", "blank"),
}
kept_joint_blocks = review_feature_reconciliation.loc[
    review_feature_reconciliation["joint_block"]
    & review_feature_reconciliation["kept_field_count"].gt(0)
]
metadata_not_predictor_gate = (
    review_feature_decision_table["lei_metadata_role"]
    .eq("diagnostic_only_not_predictor").all()
    and review_feature_decision_table["protected_metadata_role"]
    .eq("descriptive_coverage_only_not_predictor").all()
    and "lei" not in EXPECTED_REVIEW_FIELDS
    and "applicant_sex" not in EXPECTED_REVIEW_FIELDS
    and not any(
        field.startswith("applicant_race-") for field in EXPECTED_REVIEW_FIELDS
    )
)
review_screening_completion_gate = pd.Series({
    "prior_phase_4_gate": bool(
        section_phase_4_gate["ready_to_close_advanced_candidates"]
    ),
    "exactly_18_raw_fields": len(EXPECTED_REVIEW_FIELDS) == 18,
    "expected_fields_unique": len(set(EXPECTED_REVIEW_FIELDS)) == 18,
    "decision_fields_match_expected": (
        set(review_feature_decision_table["raw_field"])
        == set(EXPECTED_REVIEW_FIELDS)
        and len(review_feature_decision_table) == 18
    ),
    "audit_fields_match_expected": (
        set(review_feature_audit["raw_field"]) == set(EXPECTED_REVIEW_FIELDS)
    ),
    "every_field_has_block_id": (
        review_feature_decision_table["block_id"].notna().all()
        and review_feature_audit["block_id"].notna().all()
    ),
    "reconciled_field_count_is_18": (
        int(review_feature_reconciliation["field_count"].sum()) == 18
    ),
    "no_missing_decision": (
        review_feature_decision_table["decision"].isin(["keep", "exclude"]).all()
    ),
    "no_missing_gsv_player": (
        review_feature_decision_table["gsv_player"].notna().all()
    ),
    "excluded_maps_not_applicable": (
        review_feature_decision_table.loc[
            review_feature_decision_table["decision"].eq("exclude"),
            "gsv_player",
        ].eq("not_applicable").all()
    ),
    "kept_joint_block_shared_player": (
        kept_joint_blocks["decisions"].map(lambda values: values == ("keep",)).all()
        and kept_joint_blocks["gsv_players"].map(
            lambda values: len(values) == 1 and values[0] != "not_applicable"
        ).all()
    ),
    "special_states_preserved": (
        required_special_states.issubset(observed_special_states)
    ),
    "lei_protected_metadata_not_predictors": metadata_not_predictor_gate,
})
review_screening_completion_gate["ready_to_close_review_screening"] = bool(
    review_screening_completion_gate.all()
)

assert review_screening_completion_gate["ready_to_close_review_screening"]
display(review_feature_audit)
display(review_feature_decision_table)
display(review_feature_reconciliation)
display(review_screening_completion_gate)

,raw_field,block_id,audit_source,raw_code,description,audit_state,n,share_pct
0,conforming_loan_limit,conforming_loan_limit_status,5.1,C,Conforming,valid_informative,280700,95.7037
1,conforming_loan_limit,conforming_loan_limit_status,5.1,NC,Non-conforming,valid_informative,9436,3.2172
2,conforming_loan_limit,conforming_loan_limit_status,5.1,NA,Not applicable,valid_special,3107,1.0593
3,conforming_loan_limit,conforming_loan_limit_status,5.1,U,Undetermined,valid_special,58,0.0198
4,initially_payable_to_institution,application_submission_process,5.1,1,Initially payable to reporting institution,valid_informative,269984,92.0501
...,...,...,...,...,...,...,...,...
94,aus-4,automated_underwriting_system,5.5,3,TOTAL Scorecard,informative,4,0.0014
95,aus-4,automated_underwriting_system,5.5,5,Other AUS,other_system,1,0.0003
96,aus-5,automated_underwriting_system,5.5,,Blank later slot,blank,291536,99.3982
97,aus-5,automated_underwriting_system,5.5,1,Desktop Underwriter,informative,1268,0.4323


,raw_field,block_id,review_section,business_reason,timing_status,timing_evidence,quality_evidence,applicability_evidence,redundancy_evidence,organization_proxy_evidence,protected_group_sensitivity,screening_status,decision,decision_reason,gsv_player,validation_evidence,stability_evidence,lei_metadata_role,protected_metadata_role
0,preapproval,preapproval,5.1,Distinct conditional application concept: whet...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(4): whether the application...,100.00% valid; 100.00% informative; 0 invalid ...,All 9516 requested-preapproval rows are home p...,Conditional on home-purchase scope; not a dupl...,LEI row-weighted dominance 97.34%; global domi...,Not used as a decision rule; Phase 4 closure r...,keep_after_data_governance_review,keep,Kept because it is a distinct pre-decision app...,preapproval,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling,diagnostic_only_not_predictor,descriptive_coverage_only_not_predictor
1,submission_of_application,application_submission_process,5.1,Joint application-channel and initial institut...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(33): whether the applicant ...,100.00% valid; 97.18% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with initially_payable: full V=0.7...,LEI row-weighted dominance 96.30%; global domi...,Not used as a decision rule; block is excluded...,exclude_from_further_review,exclude,Excluded jointly because the fields primarily ...,not_applicable,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling,diagnostic_only_not_predictor,descriptive_coverage_only_not_predictor
2,initially_payable_to_institution,application_submission_process,5.1,Joint application-channel and initial institut...,available_before_decision_by_definition,"2024 HMDA FIG 4(a)(33): for an application, wh...",100.00% valid; 95.29% informative; 0 invalid c...,Codes 3 and 1111 retained separately.,Joint block with submission: full V=0.7178; in...,LEI row-weighted dominance 96.57%; global domi...,Not used as a decision rule; block is excluded...,exclude_from_further_review,exclude,Excluded jointly because the fields primarily ...,not_applicable,not_run_not_required_for_data_governance_scope,data_level_evidence_reviewed_no_model_resampling,diagnostic_only_not_predictor,descriptive_coverage_only_not_predictor
3,conforming_loan_limit,conforming_loan_limit_status,5.1,Reported conforming-loan-limit status.,not_consistently_predecision_safe,CFPB Regulation C timing: reported loan amount...,100.00% valid; 98.92% informative; 0 invalid c...,U and NA retained as separate valid special st...,Public-derived from outcome-dependent reported...,LEI row-weighted dominance 95.93%; global domi...,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded because the reported loan-amount basi...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable,diagnostic_only_not_predictor,descriptive_coverage_only_not_predictor
4,negative_amortization,negative_amortization,5.2,Contractual negative-amortization term.,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 0.14% code-...,All-cohort reporting field; code 1111 retained...,Maximum pairwise V: full=0.7072; informative-o...,LEI row-weighted dominance 99.95%; global 96.9...,Descriptive coverage only; not a decision rule...,exclude_from_further_review,exclude,Excluded for extreme rarity and concentration ...,not_applicable,not_run_not_required_for_data_governance_scope,not_applicable,diagnostic_only_not_predictor,descriptive_coverage_only_not_predictor
5,interest_only_payment,special_repayment_structure,5.2,Joint contractual special-repayment structure:...,available_before_decision_by_definition,2024 HMDA FIG 4(a)(27): contractual terms incl...,100.00% valid; 97.05% informative; 

,block_id,field_count,raw_fields,decisions,kept_field_count,excluded_field_count,gsv_players,joint_block
0,preapproval,1,"(preapproval,)","(keep,)",1,0,"(preapproval,)",False
1,application_submission_process,2,"(submission_of_application, initially_payable_...","(exclude,)",0,2,"(not_applicable,)",True
2,conforming_loan_limit_status,1,"(conforming_loan_limit,)","(exclude,)",0,1,"(not_applicable,)",False
3,negative_amortization,1,"(negative_amortization,)","(exclude,)",0,1,"(not_applicable,)",False
4,special_repayment_structure,2,"(interest_only_payment, balloon_payment)","(keep,)",2,0,"(special_repayment_structure,)",True
5,other_nonamortizing_features,1,"(other_nonamortizing_features,)","(exclude,)",0,1,"(not_applicable,)",False
6,manufactured_home_context,2,"(manufactured_home_secured_property_type, manu...","(exclude,)",0,2,"(not_applicable,)",True
7,multifamily_affordable_units,1,"(multifamily_affordable_units,)","(exclude,)",0,1,"(not_applicable,)",False
8,credit_score_model_type,2,"(applicant_credit_score_type, co-applicant_cre...","(exclude,)",0,2,"(not_applicable,)",True
9,automated_underwriting_system,5,"(aus-1, aus-2, aus-3, aus-4, aus-5)","(exclude,)",0,5,"(not_applicable,)",True


prior_phase_4_gate                       True
exactly_18_raw_fields                    True
expected_fields_unique                   True
decision_fields_match_expected           True
audit_fields_match_expected              True
every_field_has_block_id                 True
reconciled_field_count_is_18             True
no_missing_decision                      True
no_missing_gsv_player                    True
excluded_maps_not_applicable             True
kept_joint_block_shared_player           True
special_states_preserved                 True
lei_protected_metadata_not_predictors    True
ready_to_close_review_screening          True
dtype: bool

### 7.2 Final review-screening status

The reconciliation contains exactly 18 unique raw fields. Every field has one `block_id`, an explicit `keep`/`exclude` decision, and a non-null `gsv_player`. Excluded fields map to `not_applicable`; the kept repayment joint block shares `special_repayment_structure`. Special states remain separate audit states, while LEI and protected-group columns remain diagnostic metadata only.

`ready_to_close_review_screening=True`.